# Representing Research Attention as Contextually Structured Flows

Notebook accompanying the paper *Representing Research Attention as Contextually Structured Flows* (STI Conference, 2026).

We introduce the **attention flow**, a representation that situates a research output's attention in the contexts through which it is distributed: the social settings where it occurs, the language expressing it, and the time over which it unfolds.

The flow is learned with dynamic contextualised representations, in three variants along two axes, contextual and dynamic:

- **CWE** - Contextualised Word Embeddings (conditions on the linguistic context)
- **DWE** - Dynamic Word Embeddings (conditions on the social context over time)
- **DCWE** - Dynamic Contextualised Word Embeddings (both)

We evaluate the flow against altmetrics **count** and **sequence** baselines on a benchmark of analogy queries, probing three structural properties of attention:

- **Invariance** - the structure is general across outputs
- **Preservation** - the structure is stable as attention develops
- **Dependence** - the structure rests on the contexts, not on volume alone

In [1]:
import os
import re
import sys
import csv
import shutil
import json
import hmac
import hashlib
import math
import time
import requests
import pandas as pd
import numpy as np
import pickle
import torch
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import gridspec
from urllib.parse import urlencode
from pathlib import Path
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader

csv.field_size_limit(sys.maxsize)

131072

In [2]:
# project directory
project_dir = Path("..").resolve().parent

## 1. Signals

In [3]:
DATA_DIR = project_dir / "data"
df_attention_main = pd.read_csv(DATA_DIR / "df_signals_full.csv", encoding="utf-8", engine="python")
df_attention_main["count"] = 1
df_attention_main = df_attention_main.sort_values(["doi", "date", "source"]).reset_index(drop=True)
print("Per-signal attention from df_signals_full")
print(f"- Rows: {len(df_attention_main):,} | Cases: {df_attention_main['doi'].nunique():,}")
display(df_attention_main.head())

Per-signal attention from df_signals_full
- Rows: 529,461 | Cases: 113,789


,doi,title,publication_date,domain,output_type,abstract,topic_name,topic_id,source,date,signal_text,post_id,author,sentiment,count
0,10.1001/amajethics.2018.1059,How Should Clinicians Engage With Online Healt...,2018-11-01,cs,article,"Many adults, physicians, and medical students ...",internet,51,blogs,2022-09-08,Human touch and scientific veracity are missin...,https://www.kevinmd.com/2022/09/human-touch-an...,"{'name': 'KevinMD.com', 'url': 'http://www.kev...",NaN,1
1,10.1001/amajethics.2018.1059,How Should Clinicians Engage With Online Healt...,2018-11-01,cs,article,"Many adults, physicians, and medical students ...",internet,51,news,2022-12-21,The Human Touch And The Integrity Of Healthcar...,https://original.newsbreak.com/@dr-adam-tabriz...,"{'name': 'Newsbreak', 'url': 'https://www.news...",NaN,1
2,10.1001/amajethics.2018.864,How Could Commercial Terms of Use and Privacy ...,2018-09-01,cs,article,Granular personal data generated by mobile hea...,privacy,8,facebook,2019-07-26,How Could Commercial Terms of Use and Privacy ...,https://www.facebook.com/permalink.php?story_f...,"{'name': 'Alden March Bioethics Institute', 'u...",NaN,1
3,10.1001/amajethics.2018.864,How Could Commercial Terms of Use and Privacy ...,2018-09-01,cs,article,Granular personal data generated by mobile hea...,privacy,8,blogs,2023-12-19,Mental Health Apps Operating Outside Ethical B...,https://impactethics.ca/2023/12/19/mental-heal...,"{'name': 'Impact Ethics', 'url': 'https://impa...",NaN,1
4,10.1001/amajethics.2019.138,How Should Clinicians Communicate With Patient...,2019-02-01,cs,article,This commentary responds to a hypothetical cas...,artificial intelligence,12,news,2019-02-14,Viewpoint: How to tell patients AI is part of ...,https://www.beckershospitalreview.com/quality/...,"{'name': ""Becker's Hospital Review"", 'url': 'h...",NaN,1


### 1.1 Signals preprocessing

In [4]:
def preprocess_attention_df(df):
    out = df.copy()
    out["doi"] = (
        out["doi"]
        .astype(str)
        .str.strip()
        .str.replace("\u00a0", "", regex=False)
        .str.replace("\u200b", "", regex=False)
        .str.replace("\ufeff", "", regex=False)
    )
    out["source"] = (
        out["source"]
        .astype(str)
        .str.strip()
        .str.lower()
    )
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out["publication_date"] = pd.to_datetime(out["publication_date"], errors="coerce")
    out = (
        out[
            out["date"].notna()
            & out["publication_date"].notna()
            & (out["date"] >= out["publication_date"])
        ]
        .sort_values(["doi", "date", "source"])
        .reset_index(drop=True)
    )
    return out

df_attention_main = preprocess_attention_df(df_attention_main)
#df_attention_stars = preprocess_attention_df(df_attention_stars)

print("Post-publication signal tables")
print(f"- Main: {len(df_attention_main):,} rows | Cases: {df_attention_main['doi'].nunique():,}")
#print(f"- Stars: {len(df_attention_stars):,} rows | Cases: {df_attention_stars['doi'].nunique():,}")

Post-publication signal tables
- Main: 529,461 rows | Cases: 113,789


### 1.2 Signal eligibility filtering

In [5]:
MIN_MENTIONS = 10
MIN_DAYS = 3
MIN_SOURCES = 2
MIN_SOURCE_ENTROPY = 0.30
MAX_TOP_SOURCE_SHARE = 0.98
MIN_PEAK_TO_MEAN = 1.05
MIN_SPAN_DAYS = 3
MIN_PEAK_COUNT = 5
MIN_MEAN_DAILY = 1.0

In [6]:
def build_case_signal_stats(df):
    case_signal_stats = (
        df
        .groupby("doi", as_index=False)
        .agg(
            publication_date=("publication_date", "first"),
            first_attention_date=("date", "min"),
            last_attention_date=("date", "max"),
            total_mentions=("count", "sum"),
            n_days=("date", "nunique"),
            n_sources=("source", "nunique"),
        )
        .merge(
            df
            .groupby(["doi", "source"], as_index=False)
            .agg(source_mentions=("count", "sum"))
            .groupby("doi")
            .agg(
                source_entropy=("source_mentions", lambda x: (
                    -(p * np.log(p)).sum()
                    if len(p := (np.asarray(x, dtype=float) / np.asarray(x, dtype=float).sum())) > 0
                    else np.nan
                )),
                top_source_share=("source_mentions", lambda x: (
                    np.asarray(x, dtype=float).max() / np.asarray(x, dtype=float).sum()
                    if np.asarray(x, dtype=float).sum() > 0
                    else np.nan
                )),
            )
            .reset_index(),
            on="doi",
            how="left",
        )
        .merge(
            df
            .groupby(["doi", "date"], as_index=False)
            .agg(day_mentions=("count", "sum"))
            .sort_values(["doi", "date"])
            .groupby("doi", as_index=False)
            .agg(
                peak_count=("day_mentions", "max"),
                mean_daily_mentions=("day_mentions", "mean"),
                median_daily_mentions=("day_mentions", "median"),
            )
            .assign(
                peak_to_mean=lambda d: np.where(
                    d["mean_daily_mentions"] > 0,
                    d["peak_count"] / d["mean_daily_mentions"],
                    np.nan,
                )
            )[["doi", "peak_count", "mean_daily_mentions", "median_daily_mentions", "peak_to_mean"]],
            on="doi",
            how="left",
        )
    )

    case_signal_stats["span_days"] = (
        case_signal_stats["last_attention_date"] - case_signal_stats["first_attention_date"]
    ).dt.days

    return case_signal_stats


def get_eligible_case_ids(case_signal_stats):
    return case_signal_stats.loc[
        (case_signal_stats["total_mentions"] >= MIN_MENTIONS)
        & (case_signal_stats["n_days"] >= MIN_DAYS)
        & (case_signal_stats["n_sources"] >= MIN_SOURCES)
        & (case_signal_stats["source_entropy"] >= MIN_SOURCE_ENTROPY)
        & (case_signal_stats["top_source_share"] <= MAX_TOP_SOURCE_SHARE)
        & (case_signal_stats["peak_to_mean"] >= MIN_PEAK_TO_MEAN)
        & (case_signal_stats["span_days"] >= MIN_SPAN_DAYS)
        & (case_signal_stats["peak_count"] >= MIN_PEAK_COUNT)
        & (case_signal_stats["mean_daily_mentions"] >= MIN_MEAN_DAILY),
        "doi"
    ].tolist()

### 1.3 Signal aggregation

In [7]:
# 1.3 Eligibility + train / test split
from sklearn.model_selection import train_test_split

CASE_TEST_SIZE = 0.25
CASE_SPLIT_RANDOM_STATE = 42

case_signal_stats_main = build_case_signal_stats(df_attention_main)
eligible_main_case_ids = get_eligible_case_ids(case_signal_stats_main)

n_before = df_attention_main["doi"].nunique()
df_attention_main = (
    df_attention_main[df_attention_main["doi"].isin(eligible_main_case_ids)]
    .sort_values(["doi", "date", "source"])
    .reset_index(drop=True)
)
print(f"Eligibility: {df_attention_main['doi'].nunique():,} cases (from {n_before:,})\n")

main_case_ids = sorted(df_attention_main["doi"].unique())
train_case_ids, test_case_ids = train_test_split(
    main_case_ids,
    test_size=CASE_TEST_SIZE,
    random_state=CASE_SPLIT_RANDOM_STATE,
)
train_case_ids = sorted(train_case_ids)
test_case_ids = sorted(test_case_ids)

df_train = (
    df_attention_main[df_attention_main["doi"].isin(train_case_ids)]
    .sort_values(["doi", "date", "source"])
    .reset_index(drop=True)
)
df_test = (
    df_attention_main[df_attention_main["doi"].isin(test_case_ids)]
    .sort_values(["doi", "date", "source"])
    .reset_index(drop=True)
)

print("Train / test split (per-signal)")
print(f"- Eligible cases: {len(main_case_ids):,}")
print(f"- Train cases: {df_train['doi'].nunique():,} | signals: {len(df_train):,}")
print(f"- Test cases:  {df_test['doi'].nunique():,} | signals: {len(df_test):,}")
print(f"- Overlap: {len(set(df_train['doi']) & set(df_test['doi'])):,}")

Eligibility: 2,484 cases (from 113,789)

Train / test split (per-signal)
- Eligible cases: 2,484
- Train cases: 1,863 | signals: 84,492
- Test cases:  621 | signals: 28,031
- Overlap: 0


## 2. Benchmark

In [8]:
import re
from collections import Counter
from sklearn.preprocessing import RobustScaler

def _safe_entropy(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x) & (x > 0)]
    if len(x) == 0:
        return np.nan
    p = x / x.sum()
    return float(-(p * np.log(p)).sum())

def _gini(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x) & (x >= 0)]
    if len(x) == 0:
        return np.nan
    if np.allclose(x.sum(), 0):
        return 0.0
    x = np.sort(x)
    n = len(x)
    idx = np.arange(1, n + 1)
    return float((2 * np.sum(idx * x) / (n * x.sum())) - (n + 1) / n)

def _safe_share(num, den):
    return float(num / den) if pd.notna(den) and den > 0 else np.nan

def _first_rel_at(cum_share, rel_time, threshold):
    if len(cum_share) == 0 or np.all(np.isnan(cum_share)):
        return np.nan
    idx = int(np.argmax(cum_share >= threshold))
    return float(rel_time[idx])

def _relative_time_grid(n):
    if n <= 1:
        return np.array([0.0], dtype=float)
    return np.linspace(0.0, 1.0, n)

def _three_phase_masks(rel_time):
    early_mask = rel_time < (1 / 3)
    mid_mask = (rel_time >= (1 / 3)) & (rel_time < (2 / 3))
    late_mask = rel_time >= (2 / 3)
    return early_mask, mid_mask, late_mask

def _tokenize(text):
    # non-learned tokenization: lowercase, strip punctuation, split on whitespace
    if not isinstance(text, str):
        return []
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return [t for t in text.split() if t]

TEMPORAL_FEATURES = [
    "peak_time_relative",
    "peak_to_mean",
    "early_attention_share",
    "mid_attention_share",
    "late_attention_share",
    "share_before_peak",
    "share_after_peak",
    "time_concentration_entropy",
    "time_concentration_gini",
]

SOURCE_FEATURES = [
    "n_active_sources",
    "source_entropy",
    "source_gini",
    "top1_source_share",
    "top2_source_share",
    "top3_source_share",
    "mean_active_sources_per_day",
    "max_active_sources_per_day",
    "multi_source_day_share",
    "source_turnover_mean",
    "mean_new_source_share",
    "source_onset_relative",
]

LINGUISTIC_FEATURES = [
    "n_unique_terms",         
    "term_entropy",            
    "term_gini",               
    "type_token_ratio",        
    "top1_term_share",         
    "top3_term_share",         
    "mean_terms_per_signal",   
]

SCALAR_FEATURES = [
    "n_days",
    "total_mentions",
    "peak_count",
    "source_onset_span_days",
]

LOG_FEATURES = [
    "total_mentions",
    "peak_count",
    "source_onset_span_days",
    "peak_to_mean",
    "n_days",
    "n_active_sources",
    "mean_active_sources_per_day",
    "max_active_sources_per_day",
    "n_unique_terms",
    "mean_terms_per_signal",
]

print("2. Structure of Attention — helper utilities ready")

2. Structure of Attention — helper utilities ready


### 2.1 Feature Space

In [9]:
from sklearn.preprocessing import RobustScaler

def build_signal_daily(df):
    daily = (
        df.groupby(["doi", "date", "source"], as_index=False)
          .agg(count=("count", "sum"))
          .rename(columns={"doi": "case_id"})
    )
    meta = (
        df.groupby("doi", as_index=False)
          .agg(title=("title", "first"),
               topic_name=("topic_name", "first"),
               publication_date=("publication_date", "first"))
          .rename(columns={"doi": "case_id"})
    )
    out = daily.merge(meta, on="case_id", how="left")
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    return out.sort_values(["case_id", "date", "source"]).reset_index(drop=True)

REFERENCE_FEATURE_COLS = TEMPORAL_FEATURES + SOURCE_FEATURES + LINGUISTIC_FEATURES + SCALAR_FEATURES


def extract_case_temporal_features(group):
    daily = (
        group.groupby("date", as_index=False)
             .agg(total_count=("count", "sum"))
             .sort_values("date")
             .reset_index(drop=True)
    )
    counts = daily["total_count"].to_numpy(dtype=float)
    dates = pd.to_datetime(daily["date"])

    n = len(counts)
    total = counts.sum()
    mean = counts.mean() if n else np.nan
    median = np.median(counts) if n else np.nan
    peak_idx = int(np.argmax(counts)) if n else 0
    rel_time = _relative_time_grid(n)

    early_mask, mid_mask, late_mask = _three_phase_masks(rel_time)
    cum_share = np.cumsum(counts) / total if total > 0 else np.full(n, np.nan)

    start_date = dates.iloc[0] if n else pd.NaT
    end_date = dates.iloc[-1] if n else pd.NaT
    peak_date = dates.iloc[peak_idx] if n else pd.NaT
    peak_count = float(counts[peak_idx]) if n else np.nan

    return pd.Series({
        "start_date": start_date,
        "end_date": end_date,
        "n_days": int(n),
        "active_days": int((counts > 0).sum()),
        "total_mentions": float(total),

        "mean_daily_mentions": float(mean) if np.isfinite(mean) else np.nan,
        "median_daily_mentions": float(median) if np.isfinite(median) else np.nan,

        "peak_count": peak_count,
        "peak_date": peak_date,
        "peak_time_relative": float(rel_time[peak_idx]) if n else np.nan,
        "time_to_peak_days": int((peak_date - start_date).days) if n else np.nan,
        "lifespan_days": int((end_date - start_date).days) if n else np.nan,

        "peak_to_mean": (peak_count / mean) if n and pd.notna(mean) and mean > 0 else np.nan,
        "peak_to_median": (peak_count / median) if n and pd.notna(median) and median > 0 else np.nan,

        "early_attention_share": _safe_share(counts[early_mask].sum(), total),
        "mid_attention_share": _safe_share(counts[mid_mask].sum(), total),
        "late_attention_share": _safe_share(counts[late_mask].sum(), total),
        "share_before_peak": _safe_share(counts[:peak_idx].sum(), total),
        "share_after_peak": _safe_share(counts[peak_idx + 1:].sum(), total),

        "time_concentration_entropy": _safe_entropy(counts),
        "time_concentration_gini": _gini(counts),

        "cum_time_25": _first_rel_at(cum_share, rel_time, 0.25),
        "cum_time_50": _first_rel_at(cum_share, rel_time, 0.50),
        "cum_time_75": _first_rel_at(cum_share, rel_time, 0.75),
    })


def extract_source_share_features(signal_daily_df, top_k=3):
    source_totals_df = (
        signal_daily_df
        .groupby(["case_id", "source"], as_index=False)
        .agg(source_total=("count", "sum"))
    )
    ranked_df = (
        source_totals_df
        .sort_values(["case_id", "source_total"], ascending=[True, False])
        .assign(
            total_case_mentions=lambda d: d.groupby("case_id")["source_total"].transform("sum"),
            source_share=lambda d: np.where(
                d["total_case_mentions"] > 0,
                d["source_total"] / d["total_case_mentions"],
                np.nan,
            ),
            source_rank=lambda d: d.groupby("case_id").cumcount() + 1,
        )
    )
    return (
        ranked_df[ranked_df["source_rank"] <= top_k]
        .pivot(index="case_id", columns="source_rank", values="source_share")
        .rename(columns={
            1: "top1_source_share",
            2: "top2_source_share",
            3: "top3_source_share",
        })
        .reset_index()
    )


def extract_source_summary_features(signal_daily_df):
    source_totals_df = (
        signal_daily_df
        .groupby(["case_id", "source"], as_index=False)
        .agg(source_total=("count", "sum"))
    )
    return (
        source_totals_df
        .groupby("case_id", as_index=False)
        .agg(
            n_active_sources=("source", "nunique"),
            source_entropy=("source_total", _safe_entropy),
            source_gini=("source_total", _gini),
        )
    )


def extract_source_onset_features(signal_daily_df):
    onset_df = (
        signal_daily_df
        .groupby(["case_id", "source"], as_index=False)
        .agg(first_source_date=("date", "min"))
        .groupby("case_id", as_index=False)
        .agg(
            first_source_date=("first_source_date", "min"),
            last_source_onset_date=("first_source_date", "max"),
        )
    )
    onset_df["source_onset_span_days"] = (
        onset_df["last_source_onset_date"] - onset_df["first_source_date"]
    ).dt.days
    return onset_df


def build_daily_source_sets(signal_daily_df):
    return (
        signal_daily_df
        .groupby(["case_id", "date"])["source"]
        .apply(lambda x: sorted(set(x)))
        .reset_index(name="active_sources")
        .sort_values(["case_id", "date"])
        .reset_index(drop=True)
    )


def extract_source_dynamics_features(group):
    source_sets = [set(x) for x in group["active_sources"]]
    n_per_day = np.array([len(s) for s in source_sets], dtype=float)

    turnover = []
    for prev, curr in zip(source_sets[:-1], source_sets[1:]):
        union = prev | curr
        turnover.append(
            1.0 - (len(prev & curr) / len(union))
            if len(union) > 0 else 0.0
        )

    seen = set()
    new_shares = []
    for s in source_sets:
        if len(s) == 0:
            new_shares.append(0.0)
        else:
            new = s - seen
            new_shares.append(len(new) / len(s))
        seen |= s

    return pd.Series({
        "mean_active_sources_per_day": float(np.mean(n_per_day)) if len(n_per_day) else np.nan,
        "max_active_sources_per_day": float(np.max(n_per_day)) if len(n_per_day) else np.nan,
        "multi_source_day_share": float(np.mean(n_per_day > 1)) if len(n_per_day) else np.nan,
        "source_turnover_mean": float(np.mean(turnover)) if len(turnover) else 0.0,
        "mean_new_source_share": float(np.mean(new_shares)) if len(new_shares) else np.nan,
    })


def extract_linguistic_features(signal_df):
    """Per-case linguistic statistics from per-signal text (non-learned, Option A).
    Consumes the per-signal frame (doi, signal_text)."""
    rows = []
    for case_id, group in signal_df.groupby("doi"):
        per_signal_tokens = [_tokenize(t) for t in group["signal_text"].tolist()]
        per_signal_lengths = [len(toks) for toks in per_signal_tokens]
        all_tokens = [tok for toks in per_signal_tokens for tok in toks]

        total_terms = len(all_tokens)
        term_counts = Counter(all_tokens)
        counts = np.array(list(term_counts.values()), dtype=float)
        n_unique = len(term_counts)

        if total_terms > 0 and n_unique > 0:
            top_sorted = np.sort(counts)[::-1]
            top1 = float(top_sorted[0] / total_terms)
            top3 = float(top_sorted[:3].sum() / total_terms)
            ttr = float(n_unique / total_terms)
            t_entropy = _safe_entropy(counts)
            t_gini = _gini(counts)
        else:
            top1 = top3 = ttr = t_entropy = t_gini = np.nan

        rows.append({
            "case_id": case_id,
            "n_unique_terms": float(n_unique),
            "term_entropy": t_entropy,
            "term_gini": t_gini,
            "type_token_ratio": ttr,
            "top1_term_share": top1,
            "top3_term_share": top3,
            "mean_terms_per_signal": float(np.mean(per_signal_lengths)) if per_signal_lengths else np.nan,
        })
    return pd.DataFrame(rows)


def build_case_feature_table(signal_daily_df, signal_df):
    case_meta_df = (
        signal_daily_df
        .groupby("case_id", as_index=False)
        .agg(
            title=("title", "first"),
            topic_name=("topic_name", "first"),
            publication_date=("publication_date", "first"),
        )
    )

    case_temporal_df = (
        signal_daily_df
        .groupby("case_id", group_keys=False)
        .apply(extract_case_temporal_features)
        .reset_index()
    )

    source_summary_df = extract_source_summary_features(signal_daily_df)
    source_share_df = extract_source_share_features(signal_daily_df, top_k=3)
    source_onset_df = extract_source_onset_features(signal_daily_df)

    daily_source_sets_df = build_daily_source_sets(signal_daily_df)
    source_dynamics_df = (
        daily_source_sets_df
        .groupby("case_id", group_keys=False)
        .apply(extract_source_dynamics_features)
        .reset_index()
    )

    linguistic_df = extract_linguistic_features(signal_df)

    case_features_df = (
        case_meta_df
        .merge(case_temporal_df, on="case_id", how="left")
        .merge(source_summary_df, on="case_id", how="left")
        .merge(source_share_df, on="case_id", how="left")
        .merge(
            source_onset_df[
                ["case_id", "first_source_date", "last_source_onset_date", "source_onset_span_days"]
            ],
            on="case_id", how="left",
        )
        .merge(source_dynamics_df, on="case_id", how="left")
        .merge(linguistic_df, on="case_id", how="left")
        .sort_values("total_mentions", ascending=False)
        .reset_index(drop=True)
    )

    case_features_df["source_onset_relative"] = np.where(
        case_features_df["lifespan_days"] > 0,
        case_features_df["source_onset_span_days"] / case_features_df["lifespan_days"],
        0.0,
    )
    return case_features_df


def build_reference_feature_space(
    case_features_df,
    temporal_features=TEMPORAL_FEATURES,
    source_features=SOURCE_FEATURES,
    linguistic_features=LINGUISTIC_FEATURES,
    scalar_features=SCALAR_FEATURES,
    log_features=LOG_FEATURES,
    clip_min=-5,
    clip_max=5,
):
    reference_feature_cols = (
        temporal_features + source_features + linguistic_features + scalar_features
    )

    X_reference = (
        case_features_df
        .set_index("case_id")[reference_feature_cols]
        .copy()
        .replace([np.inf, -np.inf], np.nan)
        .astype(float)
    )

    for col in ["top1_source_share", "top2_source_share", "top3_source_share",
                "top1_term_share", "top3_term_share"]:
        if col in X_reference.columns:
            X_reference[col] = X_reference[col].fillna(0.0)

    for col in linguistic_features:
        if col in X_reference.columns:
            X_reference[col] = X_reference[col].fillna(0.0)

    for col in log_features:
        if col in X_reference.columns:
            X_reference[col] = np.log1p(X_reference[col])

    X_temporal = X_reference[temporal_features].copy()
    X_source = X_reference[source_features].copy()
    X_linguistic = X_reference[linguistic_features].copy()
    X_scalar = X_reference[scalar_features].copy()

    X_temporal_scaled = pd.DataFrame(
        RobustScaler().fit_transform(X_temporal),
        index=X_temporal.index, columns=X_temporal.columns,
    )
    X_source_scaled = pd.DataFrame(
        RobustScaler().fit_transform(X_source),
        index=X_source.index, columns=X_source.columns,
    )
    X_linguistic_scaled = pd.DataFrame(
        RobustScaler().fit_transform(X_linguistic),
        index=X_linguistic.index, columns=X_linguistic.columns,
    )
    X_scalar_scaled = pd.DataFrame(
        RobustScaler().fit_transform(X_scalar),
        index=X_scalar.index, columns=X_scalar.columns,
    )

    block_weight_temporal = 1 / np.sqrt(len(temporal_features))
    block_weight_source = 1 / np.sqrt(len(source_features))
    block_weight_linguistic = 1 / np.sqrt(len(linguistic_features))
    block_weight_scalar = 1 / np.sqrt(len(scalar_features))

    X_reference_scaled = pd.concat(
        [
            X_temporal_scaled * block_weight_temporal,
            X_source_scaled * block_weight_source,
            X_linguistic_scaled * block_weight_linguistic,
            X_scalar_scaled * block_weight_scalar,
        ],
        axis=1,
    ).clip(lower=clip_min, upper=clip_max)

    return {
        "X_reference": X_reference,
        "X_temporal_scaled": X_temporal_scaled,
        "X_source_scaled": X_source_scaled,
        "X_linguistic_scaled": X_linguistic_scaled,
        "X_scalar_scaled": X_scalar_scaled,
        "X_reference_scaled": X_reference_scaled,
    }

signal_daily_train_df = build_signal_daily(df_train)
case_features_df_train = build_case_feature_table(signal_daily_train_df, df_train)
reference_space_train = build_reference_feature_space(case_features_df_train)

X_reference_train = reference_space_train["X_reference"]
X_temporal_scaled_train = reference_space_train["X_temporal_scaled"]
X_source_scaled_train = reference_space_train["X_source_scaled"]
X_linguistic_scaled_train = reference_space_train["X_linguistic_scaled"]
X_scalar_scaled_train = reference_space_train["X_scalar_scaled"]
X_reference_scaled_train = reference_space_train["X_reference_scaled"]

print("Training")
print(f"- Case feature table: {case_features_df_train.shape[0]:,} cases × {case_features_df_train.shape[1]} columns")
print(f"- Temporal block: {X_temporal_scaled_train.shape[1]} features")
print(f"- Source block: {X_source_scaled_train.shape[1]} features")
print(f"- Linguistic block: {X_linguistic_scaled_train.shape[1]} features")
print(f"- Scalar block: {X_scalar_scaled_train.shape[1]} features")
print(f"- Block-balanced reference feature space: {X_reference_scaled_train.shape[0]:,} cases × {X_reference_scaled_train.shape[1]} features")
display(case_features_df_train.head())

signal_daily_test_df = build_signal_daily(df_test)
case_features_df_test = build_case_feature_table(signal_daily_test_df, df_test)
reference_space_test = build_reference_feature_space(case_features_df_test)

X_reference_test = reference_space_test["X_reference"]
X_temporal_scaled_test = reference_space_test["X_temporal_scaled"]
X_source_scaled_test = reference_space_test["X_source_scaled"]
X_linguistic_scaled_test = reference_space_test["X_linguistic_scaled"]
X_scalar_scaled_test = reference_space_test["X_scalar_scaled"]
X_reference_scaled_test = reference_space_test["X_reference_scaled"]

print("\nTest")
print(f"- Case feature table: {case_features_df_test.shape[0]:,} cases × {case_features_df_test.shape[1]} columns")
print(f"- Temporal block: {X_temporal_scaled_test.shape[1]} features")
print(f"- Source block: {X_source_scaled_test.shape[1]} features")
print(f"- Linguistic block: {X_linguistic_scaled_test.shape[1]} features")
print(f"- Scalar block: {X_scalar_scaled_test.shape[1]} features")
print(f"- Block-balanced reference feature space: {X_reference_scaled_test.shape[0]:,} cases × {X_reference_scaled_test.shape[1]} features")
display(case_features_df_test.head())

Training
- Case feature table: 1,863 cases × 50 columns
- Temporal block: 9 features
- Source block: 12 features
- Linguistic block: 7 features
- Scalar block: 4 features
- Block-balanced reference feature space: 1,863 cases × 32 features


,case_id,title,topic_name,publication_date,start_date,end_date,n_days,active_days,total_mentions,mean_daily_mentions,...,source_turnover_mean,mean_new_source_share,n_unique_terms,term_entropy,term_gini,type_token_ratio,top1_term_share,top3_term_share,mean_terms_per_signal,source_onset_relative
0,10.1038/srep40700,Neurotypical Peers are Less Willing to Interac...,social relationships,2017-02-01,2017-02-01,2026-06-22,1044,1044,1860.0,1.781609,...,0.151103,0.004151,6018.0,6.373119,0.851120,0.075738,0.021722,0.062146,42.719355,0.715869
1,10.1101/055699,Report of Partial findings from the National T...,radio,2016-05-26,2016-05-27,2024-11-06,243,243,922.0,3.794239,...,0.611531,0.022771,4506.0,6.498455,0.795305,0.122646,0.032281,0.076293,39.848156,0.261912
2,10.1038/nature16961,Mastering the game of Go with deep neural netw...,neural networks,2016-01-28,2016-01-28,2026-06-12,499,499,914.0,1.831663,...,0.689173,0.010254,6075.0,6.914733,0.745004,0.173156,0.035230,0.091124,38.385120,0.868796
3,10.1038/sdata.2016.18,The FAIR Guiding Principles for scientific dat...,data management,2016-03-15,2016-03-15,2026-06-25,504,504,799.0,1.585317,...,0.712657,0.022817,4996.0,6.814865,0.717483,0.197869,0.042299,0.106341,31.600751,0.866809
4,10.1162/qss_a_00272,The oligopoly’s shift to open access: How the ...,internet,2023-11-01,2023-11-09,2026-06-12,80,80,703.0,8.787500,...,0.514768,0.068750,1459.0,4.636465,0.902329,0.050929,0.061645,0.150272,40.751067,0.115222



Test
- Case feature table: 621 cases × 50 columns
- Temporal block: 9 features
- Source block: 12 features
- Linguistic block: 7 features
- Scalar block: 4 features
- Block-balanced reference feature space: 621 cases × 32 features


,case_id,title,topic_name,publication_date,start_date,end_date,n_days,active_days,total_mentions,mean_daily_mentions,...,source_turnover_mean,mean_new_source_share,n_unique_terms,term_entropy,term_gini,type_token_ratio,top1_term_share,top3_term_share,mean_terms_per_signal,source_onset_relative
0,10.1073/pnas.1517131113,Birds have primate-like numbers of neurons in ...,combinatorial optimization,2016-06-28,2016-07-02,2026-05-26,263,263,661.0,2.513308,...,0.578244,0.034221,3217.0,6.391424,0.774117,0.134137,0.038402,0.078139,36.282905,0.642324
1,10.1038/s41586-021-04357-7,Outracing champion Gran Turismo drivers with d...,artificial intelligence,2022-02-10,2022-02-10,2026-06-12,48,48,529.0,11.020833,...,0.547872,0.088542,1525.0,5.282260,0.858346,0.068193,0.047892,0.122166,42.274102,0.636766
2,10.1038/s41586-022-05543-x,Papers and patents are becoming less disruptiv...,education,2023-01-05,2023-01-05,2026-06-25,215,215,516.0,2.400000,...,0.704128,0.021085,4444.0,6.894949,0.688340,0.239879,0.026503,0.071737,35.903101,0.539069
3,10.1145/3065386,ImageNet classification with deep convolutiona...,neural networks,2017-05-24,2017-05-24,2026-06-06,343,343,431.0,1.256560,...,0.359649,0.016035,3041.0,6.293186,0.741277,0.163565,0.055454,0.118868,43.136891,0.728182
4,10.1073/pnas.2120481119,AI-synthesized faces are indistinguishable fro...,artificial intelligence,2022-02-22,2022-02-22,2026-05-07,113,113,381.0,3.371681,...,0.381399,0.035398,2393.0,6.375020,0.733353,0.165996,0.031909,0.079079,37.837270,0.684039


### 2.2 Structural Regimes

In [10]:
STRUCTURAL_REGIME_COL = "structural_regime"
STRUCTURAL_REGIME_MATCHES_COL = "regime_matches"
STRUCTURAL_REGIME_N_MATCHES_COL = "n_regime_matches"

STRUCTURAL_REGIMES = [
    "sequential",
    "relational",
    "distributional",
    "adversarial",
    "diffuse",
]

print("2.2 Structural Regimes")
print(f"- Regimes: {STRUCTURAL_REGIMES}")
print(f"- Assigned label column: {STRUCTURAL_REGIME_COL}")

2.2 Structural Regimes
- Regimes: ['sequential', 'relational', 'distributional', 'adversarial', 'diffuse']
- Assigned label column: structural_regime


#### 2.2.1 Regime Definitions

In [11]:
STRUCTURAL_REGIME_PRIORITY = [
    "adversarial",
    "distributional",
    "sequential",
    "diffuse",
    "relational",
]

QUANTILE_LEVELS = {
    "low": 0.25,
    "lower_mid": 0.40,
    "mid": 0.50,
    "upper_mid": 0.60,
    "high": 0.75,
    "extreme_low": 0.30,
    "extreme_high": 0.85,
}

REGIME_RULE_SPECS = {
    "sequential": [
        ("source_turnover_mean", "<=", "low"),
        ("multi_source_day_share", "<=", "mid"),
        ("time_concentration_gini", ">=", "mid"),
    ],
    "distributional": [
        ("source_turnover_mean", ">=", "high"),
        ("multi_source_day_share", ">=", "upper_mid"),
        ("time_concentration_gini", "<=", "mid"),
    ],
    "adversarial": [
        ("peak_to_mean", ">=", "extreme_high"),
        ("source_turnover_mean", ">=", "upper_mid"),
        ("late_attention_share", ">=", "upper_mid"),
    ],
    "diffuse": [
        ("peak_to_mean", "<=", "extreme_low"),
        ("time_concentration_gini", "<=", "extreme_low"),
        ("source_turnover_mean", "<=", "lower_mid"),
    ],
}

REGIME_MIN_CONDITIONS = {
    "sequential": 2,
    "distributional": 2,
    "adversarial": 2,
    "diffuse": 2,
}

def calibrate_regime_thresholds_from_train(
    case_features_df_train,
    regime_rule_specs,
    quantile_levels,
):
    thresholds = {}
    for regime, conditions in regime_rule_specs.items():
        thresholds[regime] = {}
        for feature, _operator, qualitative_level in conditions:
            quantile_value = quantile_levels[qualitative_level]
            thresholds[regime][feature] = round(
                float(case_features_df_train[feature].quantile(quantile_value)),
                3,
            )
    return thresholds

REGIME_THRESHOLDS = calibrate_regime_thresholds_from_train(
    case_features_df_train=case_features_df_train,
    regime_rule_specs=REGIME_RULE_SPECS,
    quantile_levels=QUANTILE_LEVELS,
)

def count_satisfied_conditions(row, conditions, thresholds):
    n_satisfied = 0
    for feature, operator, _qualitative_level in conditions:
        value = row[feature]
        threshold = thresholds[feature]
        if pd.isna(value):
            continue
        if operator == "<=" and value <= threshold:
            n_satisfied += 1
        elif operator == ">=" and value >= threshold:
            n_satisfied += 1
        elif operator not in {"<=", ">="}:
            raise ValueError(f"Unsupported operator: {operator}")
    return n_satisfied

def evaluate_regime_rule(row, conditions, thresholds, min_conditions):
    return count_satisfied_conditions(row, conditions, thresholds) >= min_conditions

def rule_sequential(row):
    return evaluate_regime_rule(
        row, REGIME_RULE_SPECS["sequential"], REGIME_THRESHOLDS["sequential"],
        REGIME_MIN_CONDITIONS["sequential"],
    )

def rule_distributional(row):
    return evaluate_regime_rule(
        row, REGIME_RULE_SPECS["distributional"], REGIME_THRESHOLDS["distributional"],
        REGIME_MIN_CONDITIONS["distributional"],
    )

def rule_adversarial(row):
    return evaluate_regime_rule(
        row, REGIME_RULE_SPECS["adversarial"], REGIME_THRESHOLDS["adversarial"],
        REGIME_MIN_CONDITIONS["adversarial"],
    )

def rule_diffuse(row):
    return evaluate_regime_rule(
        row, REGIME_RULE_SPECS["diffuse"], REGIME_THRESHOLDS["diffuse"],
        REGIME_MIN_CONDITIONS["diffuse"],
    )

def rule_relational(row):
    return True

STRUCTURAL_REGIME_RULES = {
    "sequential": rule_sequential,
    "relational": rule_relational,
    "distributional": rule_distributional,
    "adversarial": rule_adversarial,
    "diffuse": rule_diffuse,
}

print("2.2.1 Regime Definitions ready (Option C: strict thresholds + 2-of-3)")
print("\nQUANTILE_LEVELS:")
for level, quantile in QUANTILE_LEVELS.items():
    print(f"- {level}: {quantile:.2f}")
print("\nREGIME_MIN_CONDITIONS:")
for regime, mc in REGIME_MIN_CONDITIONS.items():
    print(f"- {regime}: {mc} of {len(REGIME_RULE_SPECS[regime])}")
print("\nREGIME_THRESHOLDS (train-calibrated):")
for regime, vals in REGIME_THRESHOLDS.items():
    print(f"- {regime}: {vals}")

2.2.1 Regime Definitions ready (Option C: strict thresholds + 2-of-3)

QUANTILE_LEVELS:
- low: 0.25
- lower_mid: 0.40
- mid: 0.50
- upper_mid: 0.60
- high: 0.75
- extreme_low: 0.30
- extreme_high: 0.85

REGIME_MIN_CONDITIONS:
- sequential: 2 of 3
- distributional: 2 of 3
- adversarial: 2 of 3
- diffuse: 2 of 3

REGIME_THRESHOLDS (train-calibrated):
- sequential: {'source_turnover_mean': 0.375, 'multi_source_day_share': 0.119, 'time_concentration_gini': 0.4}
- distributional: {'source_turnover_mean': 0.661, 'multi_source_day_share': 0.167, 'time_concentration_gini': 0.4}
- adversarial: {'peak_to_mean': 6.544, 'source_turnover_mean': 0.576, 'late_attention_share': 0.235}
- diffuse: {'peak_to_mean': 3.056, 'time_concentration_gini': 0.35, 'source_turnover_mean': 0.474}


#### 2.2.2 Regime Assignment

In [12]:
REGIME_DIAGNOSTIC_COLS = [
    "case_id",
    STRUCTURAL_REGIME_COL,
    STRUCTURAL_REGIME_MATCHES_COL,
    "time_concentration_gini",
    "peak_to_mean",
    "source_turnover_mean",
    "multi_source_day_share",
    "late_attention_share",
]

def get_structural_regime_matches(row):
    matches = []
    for regime in STRUCTURAL_REGIMES:
        if STRUCTURAL_REGIME_RULES[regime](row):
            matches.append(regime)
    return matches

def assign_structural_regime(row, priority=STRUCTURAL_REGIME_PRIORITY):
    for regime in priority:
        if STRUCTURAL_REGIME_RULES[regime](row):
            return regime
    return "relational"

def apply_structural_regime_assignment(case_features_df):
    out = case_features_df.copy()
    out[STRUCTURAL_REGIME_MATCHES_COL] = out.apply(
        get_structural_regime_matches,
        axis=1,
    )
    out[STRUCTURAL_REGIME_N_MATCHES_COL] = out[STRUCTURAL_REGIME_MATCHES_COL].apply(len)
    out[STRUCTURAL_REGIME_COL] = out.apply(
        assign_structural_regime,
        axis=1,
    )
    return out

def display_structural_regime_summary(case_features_df, split_name):
    print(f"\n{split_name}")
    print("Match-count diagnostic:")
    display(case_features_df[STRUCTURAL_REGIME_N_MATCHES_COL].value_counts().sort_index())
    print("\nAssigned regime counts:")
    display(case_features_df[STRUCTURAL_REGIME_COL].value_counts())
    print("\nAssigned regime proportions:")
    display(case_features_df[STRUCTURAL_REGIME_COL].value_counts(normalize=True).round(3))
    diagnostic_cols = [c for c in REGIME_DIAGNOSTIC_COLS if c in case_features_df.columns]
    display(case_features_df[diagnostic_cols].head(10))

case_features_df_train = apply_structural_regime_assignment(case_features_df_train)
case_features_df_test = apply_structural_regime_assignment(case_features_df_test)

case_features_df_all = pd.concat(
    [case_features_df_train, case_features_df_test],
    ignore_index=True
).drop_duplicates(subset=["case_id"])

print("2.2.2 Regime Assignment")
display_structural_regime_summary(case_features_df_train, "Training")
display_structural_regime_summary(case_features_df_test, "Test")

2.2.2 Regime Assignment

Training
Match-count diagnostic:


n_regime_matches
1    389
2    878
3    568
4     28
Name: count, dtype: int64


Assigned regime counts:


structural_regime
sequential        566
adversarial       434
relational        389
distributional    350
diffuse           124
Name: count, dtype: int64


Assigned regime proportions:


...


Assigned regime counts:


structural_regime
sequential        181
adversarial       152
relational        118
distributional    116
diffuse            54
Name: count, dtype: int64


Assigned regime proportions:


structural_regime
sequential        0.291
adversarial       0.245
relational        0.190
distributional    0.187
diffuse           0.087
Name: proportion, dtype: float64

,case_id,structural_regime,regime_matches,time_concentration_gini,peak_to_mean,source_turnover_mean,multi_source_day_share,late_attention_share
0,10.1073/pnas.1517131113,adversarial,"[sequential, relational, adversarial]",0.573287,62.467474,0.578244,0.068441,0.290469
1,10.1038/s41586-021-04357-7,relational,[relational],0.751063,8.529301,0.547872,0.145833,0.173913
2,10.1038/s41586-022-05543-x,adversarial,"[relational, distributional, adversarial]",0.521111,28.750000,0.704128,0.176744,0.193798
3,10.1145/3065386,sequential,"[sequential, relational, diffuse]",0.172032,5.570766,0.359649,0.037901,0.341067
4,10.1073/pnas.2120481119,adversarial,"[relational, adversarial]",0.571946,10.380577,0.381399,0.159292,0.317585
5,10.1038/s41467-018-06930-7,adversarial,"[relational, adversarial]",0.546579,13.059701,0.577823,0.120000,0.188060
6,10.2139/ssrn.4573321,adversarial,"[relational, adversarial]",0.428006,14.245283,0.648111,0.192053,0.327044
7,10.1093/pnasnexus/pgac110,sequential,"[sequential, relational]",0.362136,5.912052,0.339431,0.084848,0.228013
8,10.1126/sciadv.abo7019,relational,[relational],0.687135,8.140000,0.532407,0.216216,0.120000
9,10.1145/3287560.3287596,adversarial,"[relational, distributional, adversarial]",0.218688,8.170124,0.742509,0.083799,0.294606


### 2.3 Benchmark Construction

In [13]:
# 2.3 Benchmark Construction

BENCHMARK_RANDOM_STATE = 42
BENCHMARK_DISTANCE_METRIC = "euclidean"
BENCHMARK_REGIME_MODE = "within_regime"   # "within_regime" or "any"

BENCHMARK_MAX_B_PER_A = 10
BENCHMARK_MAX_C_PER_AB = 10

BENCHMARK_REGIME_COL = STRUCTURAL_REGIME_COL

print("2.3 Benchmark Construction")
print(f"- Distance metric: {BENCHMARK_DISTANCE_METRIC}")
print(f"- Regime mode: {BENCHMARK_REGIME_MODE}")
print(f"- Regime column: {BENCHMARK_REGIME_COL}")

2.3 Benchmark Construction
- Distance metric: euclidean
- Regime mode: within_regime
- Regime column: structural_regime


#### 2.3.1 Reference Feature Space

In [14]:
# 2.3.1 Reference Feature Space
benchmark_case_df = (
    case_features_df_test[
        [
            "case_id",
            BENCHMARK_REGIME_COL,
            "title",
            "topic_name",
            "publication_date",
            "total_mentions",
            "peak_time_relative",
            "peak_to_mean",
            "time_concentration_entropy",
            "time_concentration_gini",
            "source_entropy",
            "source_gini",
            "source_turnover_mean",
            "source_onset_relative",
            "multi_source_day_share",
            "late_attention_share",
        ]
    ]
    .drop_duplicates("case_id")
    .copy()
)

benchmark_case_ids = sorted(
    set(benchmark_case_df["case_id"]) & set(X_reference_scaled_test.index)
)

benchmark_case_df = (
    benchmark_case_df[benchmark_case_df["case_id"].isin(benchmark_case_ids)]
    .reset_index(drop=True)
)

X_benchmark = X_reference_scaled_test.loc[benchmark_case_ids].copy()

print(f"Benchmark case table: {benchmark_case_df.shape[0]:,} cases")
print(f"Benchmark feature space: {X_benchmark.shape[0]:,} cases × {X_benchmark.shape[1]} features")
display(benchmark_case_df.head())

Benchmark case table: 621 cases
Benchmark feature space: 621 cases × 32 features


,case_id,structural_regime,title,topic_name,publication_date,total_mentions,peak_time_relative,peak_to_mean,time_concentration_entropy,time_concentration_gini,source_entropy,source_gini,source_turnover_mean,source_onset_relative,multi_source_day_share,late_attention_share
0,10.1073/pnas.1517131113,adversarial,Birds have primate-like numbers of neurons in ...,combinatorial optimization,2016-06-28,661.0,0.503817,62.467474,4.234575,0.573287,1.201566,0.751754,0.578244,0.642324,0.068441,0.290469
1,10.1038/s41586-021-04357-7,relational,Outracing champion Gran Turismo drivers with d...,artificial intelligence,2022-02-10,529.0,0.617021,8.529301,2.703387,0.751063,0.439841,0.825851,0.547872,0.636766,0.145833,0.173913
2,10.1038/s41586-022-05543-x,adversarial,Papers and patents are becoming less disruptiv...,education,2023-01-05,516.0,0.373832,28.750000,4.533416,0.521111,1.432929,0.695560,0.704128,0.539069,0.176744,0.193798
3,10.1145/3065386,sequential,ImageNet classification with deep convolutiona...,neural networks,2017-05-24,431.0,0.488304,5.570766,5.750541,0.172032,0.852833,0.728538,0.359649,0.728182,0.037901,0.341067
4,10.1073/pnas.2120481119,adversarial,AI-synthesized faces are indistinguishable fro...,artificial intelligence,2022-02-22,381.0,0.151786,10.380577,4.022237,0.571946,0.547267,0.782902,0.381399,0.684039,0.159292,0.317585


#### 2.3.2 Pairwise Neighbours

In [15]:
# 2.3.2 Pairwise Neighbours

from sklearn.metrics import pairwise_distances

distance_matrix = pairwise_distances(
    X_benchmark.values,
    metric=BENCHMARK_DISTANCE_METRIC,
)

distance_df = pd.DataFrame(
    distance_matrix,
    index=pd.Index(X_benchmark.index, name="A"),
    columns=pd.Index(X_benchmark.index, name="B"),
)

pairwise_neighbours_df = (
    distance_df
    .stack()
    .reset_index()
)
pairwise_neighbours_df.columns = ["A", "B", "ab_dist_empirical"]

pairwise_neighbours_df = pairwise_neighbours_df[
    pairwise_neighbours_df["A"] != pairwise_neighbours_df["B"]
].copy()

pairwise_neighbours_df = pairwise_neighbours_df.merge(
    benchmark_case_df[["case_id", BENCHMARK_REGIME_COL]].rename(
        columns={"case_id": "A", BENCHMARK_REGIME_COL: "A_regime"}
    ),
    on="A",
    how="left",
).merge(
    benchmark_case_df[["case_id", BENCHMARK_REGIME_COL]].rename(
        columns={"case_id": "B", BENCHMARK_REGIME_COL: "B_regime"}
    ),
    on="B",
    how="left",
)

if BENCHMARK_REGIME_MODE == "within_regime":
    pairwise_neighbours_df = pairwise_neighbours_df[
        pairwise_neighbours_df["A_regime"] == pairwise_neighbours_df["B_regime"]
    ].copy()

pairwise_neighbours_df = (
    pairwise_neighbours_df
    .sort_values(["A", "ab_dist_empirical"], ascending=[True, True])
    .groupby("A", group_keys=False)
    .head(BENCHMARK_MAX_B_PER_A)
    .reset_index(drop=True)
)

print(pairwise_neighbours_df.groupby("A_regime").size())
print(f"Pairwise neighbour table: {pairwise_neighbours_df.shape[0]:,} rows")
display(pairwise_neighbours_df.head())

A_regime
adversarial       1520
diffuse            540
distributional    1160
relational        1180
sequential        1810
dtype: int64
Pairwise neighbour table: 6,210 rows


,A,B,ab_dist_empirical,A_regime,B_regime
0,10.1002/fee.1225,10.1016/j.patter.2021.100347,0.755780,adversarial,adversarial
1,10.1002/fee.1225,10.7861/fhj.2021-0095,0.780182,adversarial,adversarial
2,10.1002/fee.1225,10.1126/science.aat5236,0.809562,adversarial,adversarial
3,10.1002/fee.1225,10.1016/j.chb.2015.11.003,0.973551,adversarial,adversarial
4,10.1002/fee.1225,10.1016/j.ssresearch.2018.10.003,1.059036,adversarial,adversarial


#### 2.3.3 Triplet Construction

In [16]:
# 2.3.3 Triplet Construction

def build_structural_triplets(
    pairwise_neighbours_df,
    benchmark_case_ids,
    distance_df,
    benchmark_case_df,
    max_c_per_ab=10,
    regime_col=BENCHMARK_REGIME_COL,
    regime_mode="within_regime",
):
    regime_lookup = benchmark_case_df.set_index("case_id")[regime_col].to_dict()

    rows = []

    for _, pair_row in pairwise_neighbours_df.iterrows():
        a = pair_row["A"]
        b = pair_row["B"]
        ab_dist = float(pair_row["ab_dist_empirical"])
        a_regime = regime_lookup.get(a)

        if regime_mode == "within_regime":
            candidate_cs = [
                c for c in benchmark_case_ids
                if c not in {a, b} and regime_lookup.get(c) == a_regime
            ]
        else:
            candidate_cs = [c for c in benchmark_case_ids if c not in {a, b}]

        candidate_cs = sorted(candidate_cs, key=lambda c: distance_df.loc[a, c])[:max_c_per_ab]

        for c in candidate_cs:
            exclude_ids = {a, b, c}

            if regime_mode == "within_regime":
                candidate_ds = [
                    d for d in benchmark_case_ids
                    if d not in exclude_ids and regime_lookup.get(d) == a_regime
                ]
            else:
                candidate_ds = [d for d in benchmark_case_ids if d not in exclude_ids]

            if len(candidate_ds) == 0:
                continue

            d_expected = min(
                candidate_ds,
                key=lambda d: abs(distance_df.loc[c, d] - ab_dist)
            )

            target_fit_dist = float(abs(distance_df.loc[c, d_expected] - ab_dist))

            rows.append({
                "A": a,
                "B": b,
                "C": c,
                "D_expected": d_expected,
                "ab_dist_empirical": ab_dist,
                "target_fit_dist": target_fit_dist,
                "fit_ratio": target_fit_dist / (ab_dist + 1e-8),
                regime_col: a_regime,
            })

    return pd.DataFrame(rows)


triplet_candidates_df = build_structural_triplets(
    pairwise_neighbours_df=pairwise_neighbours_df,
    benchmark_case_ids=benchmark_case_ids,
    distance_df=distance_df,
    benchmark_case_df=benchmark_case_df,
    max_c_per_ab=BENCHMARK_MAX_C_PER_AB,
    regime_col=BENCHMARK_REGIME_COL,
    regime_mode=BENCHMARK_REGIME_MODE,
)

print(f"Triplet candidate table: {triplet_candidates_df.shape[0]:,} rows")
display(triplet_candidates_df.head())

Triplet candidate table: 62,100 rows


,A,B,C,D_expected,ab_dist_empirical,target_fit_dist,fit_ratio,structural_regime
0,10.1002/fee.1225,10.1016/j.patter.2021.100347,10.7861/fhj.2021-0095,10.1038/s41467-018-06930-7,0.75578,0.078582,0.103974,adversarial
1,10.1002/fee.1225,10.1016/j.patter.2021.100347,10.1126/science.aat5236,10.1073/pnas.1910837117,0.75578,0.045748,0.060531,adversarial
2,10.1002/fee.1225,10.1016/j.patter.2021.100347,10.1016/j.chb.2015.11.003,10.1038/s41467-019-14108-y,0.75578,0.011354,0.015023,adversarial
3,10.1002/fee.1225,10.1016/j.patter.2021.100347,10.1016/j.ssresearch.2018.10.003,10.1371/journal.pone.0243708,0.75578,0.164983,0.218295,adversarial
4,10.1002/fee.1225,10.1016/j.patter.2021.100347,10.1038/s41467-018-06930-7,10.7861/fhj.2021-0095,0.75578,0.078582,0.103974,adversarial


#### 2.3.4 Metadata Enrichment

In [17]:
# 2.3.4 Metadata Enrichment
def enrich_triplet_metadata(triplet_df, benchmark_case_df):
    meta_cols = [
        "case_id",
        BENCHMARK_REGIME_COL,
        "title",
        "topic_name",
        "publication_date",
        "total_mentions",
        "peak_time_relative",
        "peak_to_mean",
        "time_concentration_entropy",
        "time_concentration_gini",
        "source_entropy",
        "source_gini",
        "source_turnover_mean",
        "source_onset_relative",
        "multi_source_day_share",
        "late_attention_share",
    ]
    out = triplet_df.copy()
    for prefix in ["A", "B", "C", "D_expected"]:
        rename_map = {col: f"{prefix}_{col}" for col in meta_cols if col != "case_id"}
        rename_map["case_id"] = prefix
        tmp = benchmark_case_df[meta_cols].rename(columns=rename_map)
        out = out.merge(tmp, on=prefix, how="left")
    return out

triplet_candidates_df = enrich_triplet_metadata(
    triplet_df=triplet_candidates_df,
    benchmark_case_df=benchmark_case_df,
)

print(f"Triplet candidate table with metadata: {triplet_candidates_df.shape[0]:,} rows")
display(triplet_candidates_df.head())

Triplet candidate table with metadata: 62,100 rows


,A,B,C,D_expected,ab_dist_empirical,target_fit_dist,fit_ratio,structural_regime,A_structural_regime,A_title,...,D_expected_peak_time_relative,D_expected_peak_to_mean,D_expected_time_concentration_entropy,D_expected_time_concentration_gini,D_expected_source_entropy,D_expected_source_gini,D_expected_source_turnover_mean,D_expected_source_onset_relative,D_expected_multi_source_day_share,D_expected_late_attention_share
0,10.1002/fee.1225,10.1016/j.patter.2021.100347,10.7861/fhj.2021-0095,10.1038/s41467-018-06930-7,0.75578,0.078582,0.103974,adversarial,adversarial,Extinction of experience: the loss of human–na...,...,0.540323,13.059701,4.087678,0.546579,1.004912,0.735655,0.577823,0.376864,0.120000,0.188060
1,10.1002/fee.1225,10.1016/j.patter.2021.100347,10.1126/science.aat5236,10.1073/pnas.1910837117,0.75578,0.045748,0.060531,adversarial,adversarial,Extinction of experience: the loss of human–na...,...,0.462810,24.400000,4.162848,0.391569,1.515685,0.626667,0.732782,0.303640,0.106557,0.209524
2,10.1002/fee.1225,10.1016/j.patter.2021.100347,10.1016/j.chb.2015.11.003,10.1038/s41467-019-14108-y,0.75578,0.011354,0.015023,adversarial,adversarial,Extinction of experience: the loss of human–na...,...,0.512195,6.888889,4.593520,0.273029,1.141202,0.724691,0.552846,0.821817,0.048387,0.344444
3,10.1002/fee.1225,10.1016/j.patter.2021.100347,10.1016/j.ssresearch.2018.10.003,10.1371/journal.pone.0243708,0.75578,0.164983,0.218295,adversarial,adversarial,Extinction of experience: the loss of human–na...,...,0.000000,9.434783,4.021529,0.463041,0.914196,0.675523,0.523551,0.249444,0.107527,0.265700
4,10.1002/fee.1225,10.1016/j.patter.2021.100347,10.1038/s41467-018-06930-7,10.7861/fhj.2021-0095,0.75578,0.078582,0.103974,adversarial,adversarial,Extinction of experience: the loss of human–na...,...,0.583333,18.214286,3.594724,0.526162,0.739910,0.742857,0.523810,0.832869,0.023529,0.238095


#### 2.3.5 Quality Filtering

In [18]:
# 2.3.5 Quality Filtering

BENCHMARK_MAX_FIT_RATIO = 0.20
BENCHMARK_MAX_TARGET_FIT_DIST_QUANTILE = 0.60
BENCHMARK_MAX_QUERIES_PER_REGIME = 500  #5000
BENCHMARK_DIFFICULTY_QUANTILES = 3
BENCHMARK_RANDOM_STATE = 42

fit_dist_thr = triplet_candidates_df["target_fit_dist"].quantile(
    BENCHMARK_MAX_TARGET_FIT_DIST_QUANTILE
)

# High-quality full benchmark
structural_reasoning_benchmark_full_df = (
    triplet_candidates_df[
        (triplet_candidates_df["fit_ratio"] <= BENCHMARK_MAX_FIT_RATIO) &
        (triplet_candidates_df["target_fit_dist"] <= fit_dist_thr)
    ]
    .drop_duplicates(subset=["A", "B", "C", "D_expected"])
    .reset_index(drop=True)
)

# Balanced evaluation slice
structural_reasoning_benchmark_df = (
    structural_reasoning_benchmark_full_df
    .groupby(BENCHMARK_REGIME_COL, group_keys=False)
    .apply(
        lambda d: d.sample(
            n=min(len(d), BENCHMARK_MAX_QUERIES_PER_REGIME),
            random_state=BENCHMARK_RANDOM_STATE,
        )
    )
    .reset_index(drop=True)
)

# Difficulty stratification on the balanced benchmark
structural_reasoning_benchmark_df["difficulty"] = pd.qcut(
    structural_reasoning_benchmark_df["ab_dist_empirical"],
    q=BENCHMARK_DIFFICULTY_QUANTILES,
    labels=["easy", "medium", "hard"],
)

print(f"Full high-quality benchmark: {structural_reasoning_benchmark_full_df.shape[0]:,} queries")
print(f"Balanced evaluation benchmark: {structural_reasoning_benchmark_df.shape[0]:,} queries")

print("\nFull benchmark regime distribution:")
display(structural_reasoning_benchmark_full_df[BENCHMARK_REGIME_COL].value_counts())

print("\nBalanced benchmark regime distribution:")
display(structural_reasoning_benchmark_df[BENCHMARK_REGIME_COL].value_counts())

print("\nBalanced benchmark difficulty distribution:")
display(structural_reasoning_benchmark_df["difficulty"].value_counts())

print("\nDifficulty × regime cross-tab (row-normalised):")
display(
    pd.crosstab(
        structural_reasoning_benchmark_df["difficulty"],
        structural_reasoning_benchmark_df[BENCHMARK_REGIME_COL],
        normalize="index",
    )[["relational", "sequential", "distributional", "diffuse", "adversarial"]].round(3)
)

print("\nQuality summary — full benchmark:")
display(
    structural_reasoning_benchmark_full_df[
        ["ab_dist_empirical", "target_fit_dist", "fit_ratio"]
    ].describe().round(4)
)

print("\nQuality summary — balanced benchmark:")
display(
    structural_reasoning_benchmark_df[
        ["ab_dist_empirical", "target_fit_dist", "fit_ratio"]
    ].describe().round(4)
)

display(structural_reasoning_benchmark_df.head())

Full high-quality benchmark: 37,260 queries
Balanced evaluation benchmark: 2,500 queries

Full benchmark regime distribution:


structural_regime
sequential        11079
adversarial        8779
distributional     7344
relational         7162
diffuse            2896
Name: count, dtype: int64


Balanced benchmark regime distribution:


structural_regime
adversarial       500
diffuse           500
distributional    500
relational        500
sequential        500
Name: count, dtype: int64


Balanced benchmark difficulty distribution:


...


Quality summary — full benchmark:


,ab_dist_empirical,target_fit_dist,fit_ratio
count,37260.0000,37260.0000,37260.0000
mean,1.1899,0.0051,0.0046
std,0.2898,0.0037,0.0037
min,0.5640,0.0000,0.0000
25%,0.9912,0.0019,0.0015
50%,1.1277,0.0044,0.0037
75%,1.3237,0.0079,0.0070
max,2.7186,0.0136,0.0230



Quality summary — balanced benchmark:


,ab_dist_empirical,target_fit_dist,fit_ratio
count,2500.0000,2500.0000,2500.0000
mean,1.1939,0.0052,0.0047
std,0.2888,0.0038,0.0037
min,0.5671,0.0000,0.0000
25%,0.9947,0.0018,0.0015
50%,1.1338,0.0044,0.0038
75%,1.3199,0.0082,0.0072
max,2.6172,0.0136,0.0192


,A,B,C,D_expected,ab_dist_empirical,target_fit_dist,fit_ratio,structural_regime,A_structural_regime,A_title,...,D_expected_peak_to_mean,D_expected_time_concentration_entropy,D_expected_time_concentration_gini,D_expected_source_entropy,D_expected_source_gini,D_expected_source_turnover_mean,D_expected_source_onset_relative,D_expected_multi_source_day_share,D_expected_late_attention_share,difficulty
0,10.1016/j.isci.2020.101418,10.1038/s41598-020-77849-7,10.1073/pnas.2301974120,10.1126/science.abo0924,1.476038,0.003408,0.002309,adversarial,adversarial,Behavioral Teleporting of Individual Ethograms...,...,6.976744,2.456022,0.461628,1.109653,0.585271,0.736842,0.331912,0.050000,0.186047,hard
1,10.1038/s41467-018-04349-8,10.2139/ssrn.3140154,10.1038/s41598-020-79667-3,10.1371/journal.pgph.0000544,1.921957,0.006996,0.003640,adversarial,adversarial,Asynchronous suppression of visual cortex duri...,...,3.483871,1.873860,0.415771,0.473981,0.537634,0.583333,0.011342,0.333333,0.290323,hard
2,10.1371/journal.pone.0186621,10.1038/s41598-021-96430-4,10.1016/j.chb.2015.11.003,10.1145/3287560.3287596,1.025838,0.004294,0.004186,adversarial,adversarial,Exploring the relationship between video game ...,...,8.170124,5.037264,0.218688,1.676852,0.585477,0.742509,0.651979,0.083799,0.294606,easy
3,10.3390/ijerph19127454,10.1073/pnas.1715374115,10.1016/j.chb.2015.11.003,10.1016/j.jenvp.2018.04.001,1.210953,0.006974,0.005759,adversarial,adversarial,Effects of Indoor Plants on Human Functions: A...,...,6.923077,3.317098,0.470085,0.792037,0.634615,0.522727,0.607018,0.133333,0.259615,medium
4,10.2139/ssrn.3415726,10.1038/nature22031,10.1038/s41586-020-2442-2,10.1038/s41467-019-12552-4,0.757887,0.008909,0.011755,adversarial,adversarial,Huawei Technologies’ Links to Chinese State Se...,...,15.726027,3.829226,0.402940,1.410403,0.607877,0.674897,0.868076,0.146341,0.212329,easy


In [19]:
OUTPUT_PATH = project_dir / "data" / "structural_reasoning_benchmark_df.csv"
structural_reasoning_benchmark_df.to_csv(OUTPUT_PATH, index=False)

## 3. Representations

In [20]:
# 3. Representations
train_case_ids = set(df_train["doi"].unique())
test_case_ids = set(df_test["doi"].unique())
all_case_ids = train_case_ids | test_case_ids
df_all = pd.concat([df_train, df_test], ignore_index=True)

print(f"Training cases (representation learning): {len(train_case_ids):,}")
print(f"Test cases (evaluation only): {len(test_case_ids):,}")
print(f"All eligible cases (non-parametric baselines): {len(all_case_ids):,}")
print(f"Overlap train/test: {len(train_case_ids & test_case_ids):,}")

Training cases (representation learning): 1,863
Test cases (evaluation only): 621
All eligible cases (non-parametric baselines): 2,484
Overlap train/test: 0


### 3.1 Signal Representation

In [21]:
# 3.1 Signal Representation
# Altmetric baseline
signal_baseline_df = (
    df_test
    .groupby("doi", as_index=False)
    .agg(total_signals=("count", "sum"))
    .rename(columns={"doi": "case_id"})
    .sort_values("total_signals", ascending=False)
    .reset_index(drop=True)
)

signal_vec = {
    row.case_id: np.array([float(row.total_signals)], dtype=float)
    for row in signal_baseline_df.itertuples()
}

print("Signal representation (scalar magnitude, test partition):")
display(signal_baseline_df.head())
print("Number of cases:", signal_baseline_df["case_id"].nunique())
print("signal_vec:", len(signal_vec), "cases | dim:", next(iter(signal_vec.values())).shape[0])

Signal representation (scalar magnitude, test partition):


,case_id,total_signals
0,10.1073/pnas.1517131113,661
1,10.1038/s41586-021-04357-7,529
2,10.1038/s41586-022-05543-x,516
3,10.1145/3065386,431
4,10.1073/pnas.2120481119,381


Number of cases: 621
signal_vec: 621 cases | dim: 1


### 3.2 Sequence Representation

In [22]:
# 3.2 Sequential Representation
# Altmetric baseline
TIME_GRANULARITY = "year"        # year | month | day  (shared with the flow)
BASE_DATE = pd.to_datetime(df_test["date"]).min().normalize()

def time_index(dates):
    d = pd.to_datetime(dates)
    if TIME_GRANULARITY == "year":
        return (d.dt.year - BASE_DATE.year).astype(int)
    if TIME_GRANULARITY == "month":
        return ((d.dt.year - BASE_DATE.year) * 12 + (d.dt.month - BASE_DATE.month)).astype(int)
    if TIME_GRANULARITY == "day":
        return (d - BASE_DATE).dt.days.astype(int)
    raise ValueError(TIME_GRANULARITY)

seq_meta = df_test.copy()
seq_meta["time_idx"] = time_index(seq_meta["date"])
SEQ_LEN = int(seq_meta["time_idx"].max()) + 1

counts = seq_meta.groupby(["doi", "time_idx"])["count"].sum()
sequence_vec = {}
for doi, sub in counts.groupby(level="doi"):
    v = np.zeros(SEQ_LEN, dtype=float)
    v[sub.index.get_level_values("time_idx")] = sub.values
    sequence_vec[doi] = v

print(f"sequence_vec: {len(sequence_vec):,} cases | dim: {SEQ_LEN} | {TIME_GRANULARITY}")

sequence_vec: 621 cases | dim: 11 | year


### 3.3 Flow Embeddings

#### 3.3.1 Signal preparation

In [23]:
# 3.3.1 Data preparation — signal export
DCWE_DIR = project_dir / "data" / "dcwe"
DCWE_DIR.mkdir(parents=True, exist_ok=True)
DATA_NAME = "flowmetrics"
SOCIAL_DIM = 16

df_all = pd.concat([df_train, df_test], ignore_index=True)
BASE_DATE = pd.to_datetime(df_all["date"]).min().normalize()

def time_index(dates):
    d = pd.to_datetime(dates)
    if TIME_GRANULARITY == "year":
        return (d.dt.year - BASE_DATE.year).astype(int)
    if TIME_GRANULARITY == "month":
        return ((d.dt.year - BASE_DATE.year) * 12 + (d.dt.month - BASE_DATE.month)).astype(int)
    if TIME_GRANULARITY == "day":
        return (d - BASE_DATE).dt.days.astype(int)
    raise ValueError(TIME_GRANULARITY)

N_TIMES = int(time_index(df_all["date"]).max()) + 1
print(f"granularity={TIME_GRANULARITY} | n_times={N_TIMES}")

def to_dcwe_csv(df, split, name=DATA_NAME, out_dir=DCWE_DIR):
    d = df.copy()
    d["date"] = pd.to_datetime(d["date"], errors="coerce")
    d = d[d["date"].notna()].copy()
    out = pd.DataFrame({
        "doi": d["doi"].astype(str),
        "text": d["signal_text"].astype(str),
        "user": d["source"].astype(str),
        "time": d["date"],
        "year": d["date"].dt.year,
        "month": d["date"].dt.month,
        "day": d["date"].dt.day,
        "time_idx": time_index(d["date"]),
    })
    out = out[out["text"].str.strip().ne("") & out["text"].ne("nan")].reset_index(drop=True)
    path = out_dir / f"{name}_{split}.csv"
    out.to_csv(path, index=False)
    print(f"{split}: {len(out):,} signals, {d['doi'].nunique()} cases -> {path.name}")
    return out

DEV_FRACTION = 0.10
train_cases = sorted(df_train["doi"].unique())
rng = np.random.RandomState(42); rng.shuffle(train_cases)
n_dev = int(len(train_cases) * DEV_FRACTION)
dev_cases = set(train_cases[:n_dev])
final_train_cases = set(train_cases[n_dev:])
to_dcwe_csv(df_train[df_train["doi"].isin(final_train_cases)], "train")
to_dcwe_csv(df_train[df_train["doi"].isin(dev_cases)], "dev")
to_dcwe_csv(df_test, "test")

sources = sorted(df_all["source"].astype(str).unique())
co = (df_all[["doi", "source"]].drop_duplicates()
      .merge(df_all[["doi", "source"]].drop_duplicates(), on="doi"))
co = co[co["source_x"] < co["source_y"]]
edge_w = co.groupby(["source_x", "source_y"]).size().reset_index(name="w")
edge_set = set((r.source_x, r.source_y, int(r.w)) for r in edge_w.itertuples())
with open(DCWE_DIR / f"{DATA_NAME}_users.p", "wb") as f:
    pickle.dump(sources, f)
with open(DCWE_DIR / f"{DATA_NAME}_edges.p", "wb") as f:
    pickle.dump(edge_set, f)
print(f"\nNodes (sources): {len(sources)} | edges: {len(edge_set)}")

granularity=year | n_times=11
train: 74,659 signals, 1677 cases -> flowmetrics_train.csv
dev: 9,829 signals, 186 cases -> flowmetrics_dev.csv
test: 28,031 signals, 621 cases -> flowmetrics_test.csv

Nodes (sources): 17 | edges: 129


In [24]:
# 3.3.1 (cont.) Source graph embeddings (node2vec)
import networkx as nx
from node2vec import Node2Vec

G = nx.Graph()
G.add_nodes_from(sources)
for a, b, w in edge_set:
    G.add_edge(a, b, weight=w)

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

n2v = Node2Vec(G, dimensions=SOCIAL_DIM, walk_length=10, num_walks=200,
               weight_key="weight", workers=1, seed=42)
n2v_model = n2v.fit(window=5, min_count=1, batch_words=4)
vec_path = DCWE_DIR / f"{DATA_NAME}_vectors_{SOCIAL_DIM}.txt"
n2v_model.wv.save_word2vec_format(str(vec_path))
print(f"Saved -> {vec_path.name}")
missing = [s for s in sources if s not in n2v_model.wv]
print("sources missing a vector:", missing)

Graph: 17 nodes, 129 edges


Computing transition probabilities:   0%|          | 0/17 [00:00<?, ?it/s]

Generating walks (CPU: 1): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 200/200 [00:00<00:00, 2509.37it/s]


Saved -> flowmetrics_vectors_16.txt
sources missing a vector: []


In [25]:
# 3.3.1 Pickle the datasets for DCWE training
DCWE_REPO = project_dir / "dcwe"
sys.path.insert(0, str(DCWE_REPO / "src"))
if "data_helpers" in sys.modules:
    importlib.reload(sys.modules["data_helpers"])
from data_helpers import MLMDataset

datasets = {}
for split in ["train", "dev", "test"]:
    datasets[split] = MLMDataset(
        name=DATA_NAME,
        split=split,
        social_dim=SOCIAL_DIM,
        data_dir=str(DCWE_DIR),
    )

N_TIMES = max(max(ds.times) for ds in datasets.values()) + 1
print(f"n_times (global): {N_TIMES}\n")

for split, ds in datasets.items():
    ds.n_times = N_TIMES
    out_path = DCWE_DIR / f"mlm_{DATA_NAME}_{SOCIAL_DIM}_{split}.p"
    with open(out_path, "wb") as f:
        pickle.dump(ds, f)
    print(f"{split}: {len(ds)} examples | n_times={ds.n_times} | "
          f"vocab_filter={len(ds.filter_tensor)} -> {out_path.name}")

Token indices sequence length is longer than the specified maximum sequence length for this model (10776 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (6720 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (9769 > 512). Running this sequence through the model will result in indexing errors


n_times (global): 11

train: 74659 examples | n_times=11 | vocab_filter=10708 -> mlm_flowmetrics_16_train.p
dev: 9829 examples | n_times=11 | vocab_filter=6666 -> mlm_flowmetrics_16_dev.p
test: 28031 examples | n_times=11 | vocab_filter=9709 -> mlm_flowmetrics_16_test.p


#### 3.2.2 Training

In [26]:
# 3.3.2 Training
os.chdir(DCWE_REPO)
RESULTS_DIR = DCWE_DIR / "results"; RESULTS_DIR.mkdir(exist_ok=True)
TRAINED_DIR = DCWE_DIR / "trained"; TRAINED_DIR.mkdir(exist_ok=True)
DEVICE = -1
BATCH_SIZE = 4
LR = 1e-5
N_EPOCHS = 2
LAMBDA_A = 1e-3
LAMBDA_W = 1e-4
GNN = "gat"
!python src/main_mlm.py \
    --data_dir {DCWE_DIR} \
    --results_dir {RESULTS_DIR} \
    --trained_dir {TRAINED_DIR} \
    --data {DATA_NAME} \
    --device {DEVICE} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --n_epochs {N_EPOCHS} \
    --lambda_a {LAMBDA_A} \
    --lambda_w {LAMBDA_W} \
    --social_dim {SOCIAL_DIM} \
    --gnn {GNN}

# archive the checkpoint under a granularity-tagged name so runs don't overwrite each other
default_ckpt = TRAINED_DIR / f"mlm_{DATA_NAME}_{SOCIAL_DIM}.torch"
tagged_ckpt = TRAINED_DIR / f"mlm_{DATA_NAME}_{SOCIAL_DIM}_{TIME_GRANULARITY}.torch"
shutil.copy(default_ckpt, tagged_ckpt)
print(f"archived: {tagged_ckpt.name}")

Load training data...
Load development data...
Load test data...
Lambda a: 1e-03
Lambda w: 1e-04
Social embeddings dimensionality: 16
Number of time units: 11
Number of vocabulary items: 10708
Best perplexity so far: 8.457237226402137
Train model...
Processed 0 examples...
Processed 4000 examples...
Processed 8000 examples...
Processed 12000 examples...
Processed 16000 examples...
Processed 20000 examples...
Processed 24000 examples...
Processed 28000 examples...
Processed 32000 examples...
Processed 36000 examples...
Processed 40000 examples...
Processed 44000 examples...
Processed 48000 examples...
Processed 52000 examples...
Processed 56000 examples...
Processed 60000 examples...
Processed 64000 examples...
Processed 68000 examples...
Processed 72000 examples...
Evaluate model...
8.465003593451614 9.045784450246211
Processed 0 examples...
Processed 4000 examples...
Processed 8000 examples...
Processed 12000 examples...
Processed 16000 examples...
Processed 20000 examples...
Processe

#### 3.3.3 Embeddings

In [27]:
# 3.3.3 Embeddings
# Per-signal vectors from the trained model, three modes (one forward call, three taps):
#   dcwe = static + offset -> BERT  (full)
#   cwe  = static -> BERT           (contextualised only)
#   dwe  = static + offset, no BERT (dynamic only)
sys.path.insert(0, str(DCWE_REPO / "src"))
from data_helpers import MLMDataset, MLMCollator
from model import MLMModel
device = torch.device("cpu")
TRAINED_DIR = DCWE_DIR / "trained"
def load_dcwe_model(ds):
    model = MLMModel(n_times=ds.n_times, social_dim=SOCIAL_DIM, gnn="gat")
    ckpt = TRAINED_DIR / f"mlm_{DATA_NAME}_{SOCIAL_DIM}_{TIME_GRANULARITY}.torch"
    state = torch.load(ckpt, map_location=device)
    model.load_state_dict(state)
    return model.to(device).eval()
def extract_vectors(ds, model, mode="dcwe"):
    graph_data = ds.graph_data.to(device)
    vocab_filter = ds.filter_tensor.to(device)
    collator = MLMCollator(ds.user2id, ds.tok)
    loader = DataLoader(ds, batch_size=8, collate_fn=collator, shuffle=False)
    vecs = []
    with torch.no_grad():
        for batch in loader:
            labels, users, times, years, months, days, reviews, masks, segs = batch
            users = users.to(device); reviews = reviews.to(device)
            masks = masks.to(device); segs = segs.to(device)
            bert_embs, input_embs = model(labels, reviews, masks, segs, users,
                                          graph_data, times, vocab_filter, embs_only=True)
            if mode == "dwe":
                emb = input_embs
            else:
                src = bert_embs if mode == "cwe" else input_embs
                outputs = model.bert.bert(inputs_embeds=src, attention_mask=masks,
                                          token_type_ids=segs, output_hidden_states=True)
                emb = torch.stack(outputs.hidden_states[1:7], dim=0).mean(dim=0)
            m = masks.unsqueeze(-1).float()
            pooled = (emb * m).sum(1) / m.sum(1).clamp(min=1e-9)
            vecs.append(pooled.cpu().numpy())
    return np.vstack(vecs)
SPLIT = "test"
with open(DCWE_DIR / f"mlm_{DATA_NAME}_{SOCIAL_DIM}_{SPLIT}.p", "rb") as f:
    ds = pickle.load(f)
csv = pd.read_csv(DCWE_DIR / f"{DATA_NAME}_{SPLIT}.csv", parse_dates=["time"])
csv = csv.dropna().reset_index(drop=True)
assert len(csv) == len(ds), f"alignment mismatch: csv {len(csv)} vs ds {len(ds)}"
model = load_dcwe_model(ds)
vectors_dcwe = extract_vectors(ds, model, mode="dcwe")
vectors_cwe  = extract_vectors(ds, model, mode="cwe")
vectors_dwe  = extract_vectors(ds, model, mode="dwe")
signal_dois = csv["doi"].tolist()
print(f"loaded model: mlm_{DATA_NAME}_{SOCIAL_DIM}_{TIME_GRANULARITY}.torch")
print("per-signal vectors:")
print(f"- dcwe: {vectors_dcwe.shape} | cwe: {vectors_cwe.shape} | dwe: {vectors_dwe.shape}")
print(f"- aligned dois: {len(signal_dois)} | unique outputs: {csv['doi'].nunique()}")

BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'c

loaded model: mlm_flowmetrics_16_year.torch
per-signal vectors:
- dcwe: (28031, 768) | cwe: (28031, 768) | dwe: (28031, 768)
- aligned dois: 28031 | unique outputs: 621


#### 3.3.4 Flow representation

In [28]:
# 3.3.4 Flow representation
from sklearn.decomposition import PCA

assert "vectors_dcwe" in globals() and "csv" in globals(), "run 3.3.3 first"
assert len(csv) == len(vectors_dcwe), "alignment broken — re-run 3.3.3"

FLOW_MODE = "mean_time"          # mean_time | weighted | summary | trajectory_pca
PCA_DIM = 128

signal_meta = csv.copy()
signal_meta["time_idx"] = signal_meta["time_idx"].astype(int)
N_BINS = int(signal_meta["time_idx"].max()) + 1

def _aggregate(vecs, times, mode):
    uniq = sorted(set(times.tolist()))
    C = np.stack([vecs[times == t].mean(axis=0) for t in uniq], axis=0)   # (n_obs_times, 768)
    if mode == "mean_time":
        return C.mean(axis=0)
    if mode == "weighted":
        w = np.asarray(uniq, dtype=float) + 1.0
        return (C * w[:, None]).sum(axis=0) / w.sum()
    if mode == "summary":
        first, last, mean = C[0], C[-1], C.mean(axis=0)
        drift = np.diff(C, axis=0).mean(axis=0) if len(C) > 1 else np.zeros_like(mean)
        return np.concatenate([first, last, mean, drift])
    if mode == "trajectory_grid":
        grid = np.zeros((N_BINS, C.shape[1]), dtype=float)
        for t, c in zip(uniq, C):
            grid[t] = c
        return grid.reshape(-1)
    raise ValueError(mode)

def build_flow(vectors, mode=FLOW_MODE):
    groups = {doi: sub.index.to_numpy() for doi, sub in signal_meta.groupby("doi")}
    times_all = signal_meta["time_idx"].values
    if mode == "trajectory_pca":
        dois = list(groups)
        X = np.vstack([_aggregate(vectors[idx], times_all[idx], "trajectory_grid") for idx in groups.values()])
        Xr = PCA(n_components=min(PCA_DIM, *X.shape), random_state=42).fit_transform(X)
        return {doi: Xr[i] for i, doi in enumerate(dois)}
    return {doi: _aggregate(vectors[idx], times_all[idx], mode) for doi, idx in groups.items()}

dcwe_vec = build_flow(vectors_dcwe)
cwe_vec  = build_flow(vectors_cwe)
dwe_vec  = build_flow(vectors_dwe)
print(f"FLOW_MODE={FLOW_MODE} | dcwe: {len(dcwe_vec):,} | cwe: {len(cwe_vec):,} | dwe: {len(dwe_vec):,} "
      f"| dim: {next(iter(dcwe_vec.values())).shape[0]}")

FLOW_MODE=mean_time | dcwe: 621 | cwe: 621 | dwe: 621 | dim: 768


## 4. Evaluation

In [29]:
# 4. Evaluation — setup
QUAL_OPERATIONS = {
    "Linear translation":      "offset",            # D ~= A - B + C
    "Relational similarity":   "difference_match",  # (A - B) ~= (D - C)
    "Counterfactual transfer": "counterfactual",    # D ~= C + (B - A)
}
qual_representations = {
    "signal":   signal_vec,
    "sequence": sequence_vec,
    "cwe":      cwe_vec,
    "dwe":      dwe_vec,
    "dcwe":     dcwe_vec,
}
bench_ids = set(structural_reasoning_benchmark_df[["A", "B", "C", "D_expected"]].values.ravel())
for m, rep in qual_representations.items():
    missing = bench_ids - set(rep)
    assert not missing, f"{m} missing {len(missing)} benchmark cases"
print("4. Evaluation — setup")
print(f"- benchmark cases: {len(bench_ids):,} | queries: {len(structural_reasoning_benchmark_df):,}")
for m, rep in qual_representations.items():
    print(f"- {m}: dim={next(iter(rep.values())).shape[0]}, cases={len(rep)}")

4. Evaluation — setup
- benchmark cases: 621 | queries: 2,500
- signal: dim=1, cases=621
- sequence: dim=11, cases=621
- cwe: dim=768, cases=621
- dwe: dim=768, cases=621
- dcwe: dim=768, cases=621


### 4.1 Operators

In [40]:
# 4.1 Operators
def op_offset(a, b, c):
    return a - b + c                  # D ~= A - B + C

def op_difference_match(a, b, c):
    return a - b, c                   # (D - C) to (A - B)

def op_counterfactual(a, b, c):
    return c + (b - a)                # D ~= C + (B - A)

QUAL_OP = {
    "offset": op_offset,
    "difference_match": op_difference_match,
    "counterfactual": op_counterfactual,
}
print("4.1 Operators ready:", list(QUAL_OP.keys()))

4.1 Operators ready: ['offset', 'difference_match', 'counterfactual']


### 4.2 Operations

In [41]:
# 4.2 Ranking and metrics
RECALL_KS = [10, 50, 100]

def metric_cosine(x, y):
    x = np.asarray(x, dtype=float).ravel()
    y = np.asarray(y, dtype=float).ravel()
    nx, ny = np.linalg.norm(x), np.linalg.norm(y)
    if nx == 0 or ny == 0:
        return np.inf
    return float(1.0 - np.dot(x, y) / (nx * ny))

def _rank_row(a, b, c, d_exp, rank, n, op, regime, difficulty, retrieved):
    """Shared output schema: MRR fields plus whole-ranking fields (percentile, recall@k)."""
    row = {
        "A": a, "B": b, "C": c, "D_expected": d_exp,
        "operation": op, "structural_regime": regime, "difficulty": difficulty,
        "n_candidates": n, "retrieved_D": retrieved,
        "expected_rank": rank,
        "reciprocal_rank": 1.0 / rank,
        "percentile": (rank - 1) / (n - 1) if n > 1 else 0.0,   # 0=top, 1=bottom; <0.5 beats random
        "top1_hit": int(rank == 1),
        "top5_hit": int(rank <= 5),
    }
    for k in RECALL_KS:
        row[f"recall_at_{k}"] = int(rank <= k)
    return row

def rank_expected_candidate(triplet_df, lookup, operation_name="offset", query_lookup=None):
    op_fn = QUAL_OP[operation_name]
    q = query_lookup if query_lookup is not None else lookup
    rows = []
    for _, row in triplet_df.iterrows():
        a, b, c, d_exp = row["A"], row["B"], row["C"], row["D_expected"]
        obj_a, obj_b, obj_c = q[a], q[b], q[c]
        exclude = {a, b, c}
        if operation_name == "difference_match":
            transform, base = op_fn(obj_a, obj_b, obj_c)
            def cand_dist(obj_d): return metric_cosine(transform, obj_d - base)
        else:
            target = op_fn(obj_a, obj_b, obj_c)
            def cand_dist(obj_d): return metric_cosine(target, obj_d)
        dists = [(d, cand_dist(obj_d)) for d, obj_d in lookup.items() if d not in exclude]
        dists.sort(key=lambda t: t[1])
        ranked_ids = [d for d, _ in dists]
        rank = ranked_ids.index(d_exp) + 1
        rows.append(_rank_row(a, b, c, d_exp, rank, len(ranked_ids), operation_name,
                              row.get("structural_regime"), row.get("difficulty"), ranked_ids[0]))
    return pd.DataFrame(rows)

def rank_random(triplet_df, candidates, n_seeds=10, seed=0):
    """Empirical chance floor, averaged over n_seeds random orderings to reduce noise.
    Same pool and exclusions as the real retrieval."""
    cand_ids = list(candidates)
    rows = []
    for _, row in triplet_df.iterrows():
        a, b, c, d_exp = row["A"], row["B"], row["C"], row["D_expected"]
        pool = [x for x in cand_ids if x not in {a, b, c}]
        n = len(pool)
        d_pos = pool.index(d_exp)
        rr, pct = [], []
        for s in range(n_seeds):
            rng = np.random.default_rng(seed + s)
            rank = int(np.where(rng.permutation(n) == d_pos)[0][0]) + 1
            rr.append(1.0 / rank)
            pct.append((rank - 1) / (n - 1) if n > 1 else 0.0)
        rows.append({
            "A": a, "B": b, "C": c, "D_expected": d_exp,
            "operation": "random",
            "structural_regime": row.get("structural_regime"),
            "difficulty": row.get("difficulty"),
            "n_candidates": n,
            "reciprocal_rank": float(np.mean(rr)),
            "percentile": float(np.mean(pct)),
        })
    return pd.DataFrame(rows)

def summarise_mrr(rank_df, group_cols):
    if rank_df.empty:
        return pd.DataFrame()
    agg = {
        "n_queries": ("A", "count"),
        "mrr": ("reciprocal_rank", "mean"),
        "mean_percentile": ("percentile", "mean"),
        "top1": ("top1_hit", "mean"),
        "top5": ("top5_hit", "mean"),
        "mean_rank": ("expected_rank", "mean"),
    }
    for k in RECALL_KS:
        agg[f"recall_at_{k}"] = (f"recall_at_{k}", "mean")
    return rank_df.groupby(group_cols, as_index=False).agg(**agg)

def mrr_ci(rank_df, n_boot=2000, alpha=0.05, seed=0):
    """Bootstrap CI for MRR by resampling queries."""
    rr = rank_df["reciprocal_rank"].to_numpy()
    rng = np.random.default_rng(seed)
    boots = rng.choice(rr, size=(n_boot, len(rr)), replace=True).mean(axis=1)
    lo, hi = np.quantile(boots, [alpha / 2, 1 - alpha / 2])
    return {"mrr": float(rr.mean()), "ci_low": float(lo), "ci_high": float(hi)}

def paired_mrr_test(rank_df_a, rank_df_b, n_boot=2000, seed=0):
    """Paired bootstrap on per-query reciprocal-rank differences (a - b).
    Both frames must cover the same queries in the same order."""
    rr_a = rank_df_a["reciprocal_rank"].to_numpy()
    rr_b = rank_df_b["reciprocal_rank"].to_numpy()
    diff = rr_a - rr_b
    rng = np.random.default_rng(seed)
    boots = rng.choice(diff, size=(n_boot, len(diff)), replace=True).mean(axis=1)
    p = 2 * min((boots <= 0).mean(), (boots >= 0).mean())
    return {"delta_mrr": float(diff.mean()), "p_value": float(min(p, 1.0))}

print("4.2 Ranking and metrics ready")

4.2 Ranking and metrics ready


## 5. Evaluation Pipeline

In [42]:
# 5. Evaluation Pipeline — setup
benchmark = structural_reasoning_benchmark_df.copy()
shared_ids = set.intersection(*(set(rep) for rep in qual_representations.values()))
REPS = {m: {cid: rep[cid] for cid in shared_ids} for m, rep in qual_representations.items()}
print(f"benchmark queries: {len(benchmark):,} | shared cases: {len(shared_ids):,}")

benchmark queries: 2,500 | shared cases: 621


### 5.1 Structural invariance

In [43]:
# 5.1 Structural invariance
invariance_ranks = []
for m, rep in REPS.items():
    rk = rank_expected_candidate(benchmark, rep, operation_name="offset")
    rk["model"] = m
    invariance_ranks.append(rk)
rk_rand = rank_random(benchmark, REPS["dcwe"])
rk_rand["model"] = "random"
invariance_ranks.append(rk_rand)
invariance_ranks = pd.concat(invariance_ranks, ignore_index=True)

order = list(REPS) + ["random"]

def regime_table(value_col):
    t = (invariance_ranks.groupby(["model", "structural_regime"])[value_col]
         .mean().unstack("structural_regime"))
    t["Overall"] = invariance_ranks.groupby("model")[value_col].mean()
    return t.reindex(order).round(3)

print("MRR (top-heavy):")
display(regime_table("reciprocal_rank"))
print("\nmean_percentile (whole-ranking; <0.5 beats random, lower is better):")
display(regime_table("percentile"))

n = len(REPS["dcwe"]) - 3
print(f"\ntheoretical chance: MRR ≈ ln(N)/N = {np.log(n)/n:.4f} | mean_percentile = 0.500  (N={n})")

MRR (top-heavy):


structural_regime,adversarial,diffuse,distributional,relational,sequential,Overall
model,,,,,,
signal,0.011,0.011,0.008,0.010,0.007,0.009
sequence,0.014,0.012,0.012,0.015,0.014,0.013
cwe,0.029,0.016,0.008,0.010,0.010,0.014
dwe,0.024,0.016,0.009,0.011,0.012,0.014
dcwe,0.030,0.017,0.009,0.010,0.010,0.015
random,0.010,0.012,0.011,0.012,0.010,0.011



mean_percentile (whole-ranking; <0.5 beats random, lower is better):


structural_regime,adversarial,diffuse,distributional,relational,sequential,Overall
model,,,,,,
signal,0.491,0.446,0.466,0.519,0.530,0.490
sequence,0.465,0.455,0.526,0.502,0.504,0.490
cwe,0.338,0.451,0.483,0.514,0.517,0.461
dwe,0.328,0.432,0.544,0.481,0.527,0.462
dcwe,0.323,0.450,0.487,0.511,0.521,0.458
random,0.508,0.500,0.496,0.496,0.512,0.502



theoretical chance: MRR ≈ ln(N)/N = 0.0104 | mean_percentile = 0.500  (N=618)


### 5.2 Structural preservation

In [44]:
# 5.2 Structural preservation — setup (reusable aggregation + truncation)
doi_rows = {doi: sub for doi, sub in df_test.groupby("doi")}
doi_vecidx = {doi: sub.index.to_numpy() for doi, sub in signal_meta.groupby("doi")}
FLOW_VECTORS = {"cwe": vectors_cwe, "dwe": vectors_dwe, "dcwe": vectors_dcwe}

def build_signal_rep(rows):
    return np.array([float(rows["count"].sum())], dtype=float)

def build_sequence_rep(rows):
    v = np.zeros(SEQ_LEN, dtype=float)
    np.add.at(v, time_index(rows["date"]).values, rows["count"].values)
    return v

def truncate_rep(doi, frac, model):
    if model in FLOW_VECTORS:
        idx = doi_vecidx[doi]
        order = np.argsort(signal_meta["time"].values[idx], kind="stable")
        k = max(1, int(round(len(idx) * frac)))
        kept = idx[order[:k]]
        return _aggregate(FLOW_VECTORS[model][kept], signal_meta["time_idx"].values[kept], FLOW_MODE)
    rows = doi_rows[doi].sort_values("date")
    k = max(1, int(round(len(rows) * frac)))
    rows = rows.iloc[:k]
    return build_signal_rep(rows) if model == "signal" else build_sequence_rep(rows)

TRUNCATIONS = [0.25, 0.50, 0.75]

In [45]:
# 5.2 Structural preservation — run
preservation_rows = []
for frac in TRUNCATIONS:
    for m in REPS:
        trunc_lookup = {doi: truncate_rep(doi, frac, m) for doi in REPS[m]}
        df = rank_expected_candidate(benchmark, REPS[m], query_lookup=trunc_lookup)
        df["model"], df["frac"] = m, frac
        preservation_rows.append(df)
    rk = rank_random(benchmark, REPS["dcwe"])
    rk["model"], rk["frac"] = "random", frac
    preservation_rows.append(rk)
preservation_ranks = pd.concat(preservation_rows, ignore_index=True)

def preservation_table(value_col):
    per_regime = preservation_ranks.groupby(["model", "structural_regime", "frac"])[value_col].mean()
    overall = preservation_ranks.groupby(["model", "frac"])[value_col].mean()
    overall.index = pd.MultiIndex.from_arrays(
        [overall.index.get_level_values("model"),
         ["Overall"] * len(overall),
         overall.index.get_level_values("frac")],
        names=["model", "structural_regime", "frac"])
    combined = pd.concat([per_regime, overall])
    regimes = sorted(preservation_ranks["structural_regime"].unique()) + ["Overall"]
    return (
        combined.unstack("frac")
        .rename(columns={0.25: "25%", 0.50: "50%", 0.75: "75%"})
        .reindex(pd.MultiIndex.from_product([list(REPS) + ["random"], regimes],
                                            names=["model", "structural_regime"]))
        .round(3)
    )

print("MRR (top-heavy):")
display(preservation_table("reciprocal_rank"))
print("\nmean_percentile (whole-ranking; <0.5 beats random, lower is better):")
display(preservation_table("percentile"))

MRR (top-heavy):


frac                          25%    50%    75%
model    structural_regime                     
signal   adversarial        0.011  0.011  0.011
         diffuse            0.011  0.011  0.011
         distributional     0.008  0.008  0.008
         relational         0.010  0.010  0.010
         sequential         0.007  0.007  0.007
         Overall            0.009  0.009  0.009
sequence adversarial        0.009  0.009  0.010
         diffuse            0.011  0.014  0.014
         distributional     0.016  0.017  0.015
         relational         0.016  0.015  0.014
         sequential         0.010  0.013  0.014
         Overall            0.012  0.014  0.013
cwe      adversarial        0.023  0.021  0.028
         diffuse            0.011  0.014  0.017
         distributional     0.011  0.010  0.010
         relational         0.013  0.014  0.017
         sequential         0.013  0.014  0.013
         Overall            0.014  0.015  0.017
dwe      adversarial        0.012  0.015  0.020
         diffuse            0.011  0.011  0.011
         distributional     0.013  0.011  0.013
         relational         0.019  0.020  0.018
         sequential         0.012  0.013  0.014
         Overall            0.013  0.014  0.015
dcwe     adversarial        0.021  0.020  0.024
         diffuse            0.013  0.013  0.017
         distributional     0.011  0.010  0.010
         relational         0.013  0.016  0.015
         sequential         0.013  0.016  0.012
         Overall            0.014  0.015  0.016
random   adversarial        0.010  0.010  0.010
         diffuse            0.012  0.012  0.012
         distributional     0.011  0.011  0.011
         relational         0.012  0.012  0.012
         sequential         0.010  0.010  0.010
         Overall            0.011  0.011  0.011


mean_percentile (whole-ranking; <0.5 beats random, lower is better):


frac                          25%    50%    75%
model    structural_regime                     
signal   adversarial        0.491  0.491  0.491
         diffuse            0.446  0.446  0.446
         distributional     0.466  0.466  0.466
         relational         0.519  0.519  0.519
         sequential         0.530  0.530  0.530
         Overall            0.490  0.490  0.490
sequence adversarial        0.467  0.466  0.472
         diffuse            0.474  0.465  0.455
         distributional     0.512  0.517  0.515
         relational         0.505  0.501  0.501
         sequential         0.507  0.501  0.497
         Overall            0.493  0.490  0.488
cwe      adversarial        0.345  0.332  0.332
         diffuse            0.477  0.468  0.475
         distributional     0.484  0.484  0.488
         relational         0.524  0.528  0.524
         sequential         0.526  0.521  0.513
         Overall            0.471  0.467  0.466
dwe      adversarial        0.355  0.351  0.343
         diffuse            0.449  0.439  0.436
         distributional     0.528  0.534  0.541
         relational         0.479  0.481  0.486
         sequential         0.538  0.531  0.531
         Overall            0.470  0.467  0.467
dcwe     adversarial        0.332  0.319  0.318
         diffuse            0.472  0.466  0.469
         distributional     0.486  0.485  0.491
         relational         0.522  0.524  0.522
         sequential         0.530  0.525  0.519
         Overall            0.469  0.464  0.464
random   adversarial        0.508  0.508  0.508
         diffuse            0.500  0.500  0.500
         distributional     0.496  0.496  0.496
         relational         0.496  0.496  0.496
         sequential         0.512  0.512  0.512
         Overall            0.502  0.502  0.502

### 5.3 Structural dependence

In [50]:
# 5.3 Structural dependence — setup
rng_perturb = np.random.default_rng(42)
_CTX = [["time", "year", "month", "day", "time_idx", "date"], ["user", "source"], ["text", "signal_text"]]

def _perm_across(df, cols):
    out = df.copy()
    cols = [c for c in cols if c in out.columns]
    out.loc[:, cols] = out.iloc[rng_perturb.permutation(len(out))][cols].values
    return out

def _perm_across_joint(df):
    # reassign all contexts (temporal, social, linguistic) across outputs
    out = df.copy()
    for cols in _CTX:
        cols = [c for c in cols if c in out.columns]
        if cols:
            out.loc[:, cols] = out.iloc[rng_perturb.permutation(len(out))][cols].values
    return out

def identity(df): return df.copy()

PERTURBATIONS = {
    "temporal_across":   lambda df: _perm_across(df, _CTX[0]),
    "social_across":     lambda df: _perm_across(df, _CTX[1]),
    "linguistic_across": lambda df: _perm_across(df, _CTX[2]),
    "across":            _perm_across_joint,
}

def reembed_flows(perturbed_df):
    tmp = DCWE_DIR / f"{DATA_NAME}_perturbed.csv"
    perturbed_df.to_csv(tmp, index=False)
    ds_p = MLMDataset(name=DATA_NAME, split="perturbed", social_dim=SOCIAL_DIM, data_dir=str(DCWE_DIR))
    meta = pd.read_csv(tmp, parse_dates=["time"]).dropna().reset_index(drop=True)
    groups = {doi: sub.index.to_numpy() for doi, sub in meta.groupby("doi")}
    times = meta["time_idx"].astype(int).values
    flows = {}
    for mode in ("cwe", "dwe", "dcwe"):
        vecs = extract_vectors(ds_p, model, mode=mode)
        flows[mode] = {doi: _aggregate(vecs[idx], times[idx], FLOW_MODE) for doi, idx in groups.items()}
    return flows

def perturbed_reps(perturb_fn):
    flows = reembed_flows(perturb_fn(csv))
    pdata = perturb_fn(df_test)
    signal_p, sequence_p = {}, {}
    for doi, sub in pdata.groupby("doi"):
        signal_p[doi] = np.array([float(sub["count"].sum())])
        v = np.zeros(SEQ_LEN); np.add.at(v, time_index(sub["date"]).values, sub["count"].values)
        sequence_p[doi] = v
    return {"signal": signal_p, "sequence": sequence_p, **flows}

In [51]:
# 5.3 Structural dependence — run
frames = []
reps_orig = perturbed_reps(identity)
for m in REPS:
    df = rank_expected_candidate(benchmark, REPS[m], query_lookup=reps_orig[m])
    df["model"], df["perturbation"] = m, "original"
    frames.append(df)

reps_p = perturbed_reps(PERTURBATIONS["across"])
for m in REPS:
    df = rank_expected_candidate(benchmark, REPS[m], query_lookup=reps_p[m])
    df["model"], df["perturbation"] = m, "across"
    frames.append(df)

_rand = rank_random(benchmark, REPS["dcwe"])
for tag in ["original", "across"]:
    r = _rand.copy()
    r["model"], r["perturbation"] = "random", tag
    frames.append(r)

dependence_ranks = pd.concat(frames, ignore_index=True)

order = list(REPS) + ["random"]
regime_order = ["sequential", "distributional", "adversarial", "diffuse", "relational"]

def dependence_table():
    om = dependence_ranks.perturbation == "original"
    pm = dependence_ranks.perturbation == "across"
    orig_r = dependence_ranks[om].groupby(["model", "structural_regime"])["percentile"].mean()
    pert_r = dependence_ranks[pm].groupby(["model", "structural_regime"])["percentile"].mean()
    orig_o = dependence_ranks[om].groupby("model")["percentile"].mean()
    pert_o = dependence_ranks[pm].groupby("model")["percentile"].mean()
    rows = []
    for m in order:
        for rg in regime_order:
            rows.append((m, rg, orig_r.get((m, rg)), pert_r.get((m, rg))))
        rows.append((m, "Overall", orig_o.get(m), pert_o.get(m)))
    t = pd.DataFrame(rows, columns=["model", "regime", "MPR (Original)", "MPR (Perturbed)"])
    t["ΔMPR"] = t["MPR (Perturbed)"] - t["MPR (Original)"]
    t["ΔMPR%"] = (t["MPR (Perturbed)"] - t["MPR (Original)"]) / t["MPR (Original)"] * 100
    return t.set_index(["model", "regime"]).round(3)

print("Structural dependence — MPR before and after across-output context perturbation (ΔMPR > 0 = depends on contexts):")
display(dependence_table())

Token indices sequence length is longer than the specified maximum sequence length for this model (9769 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (9748 > 512). Running this sequence through the model will result in indexing errors


Structural dependence — MPR before and after across-output context perturbation (ΔMPR > 0 = depends on contexts):


MPR (Original)  MPR (Perturbed)   ΔMPR   ΔMPR%
model    regime                                                        
signal   sequential               0.530            0.530  0.000   0.000
         distributional           0.466            0.466  0.000   0.000
         adversarial              0.491            0.491  0.000   0.000
         diffuse                  0.446            0.446  0.000   0.000
         relational               0.519            0.519  0.000   0.000
         Overall                  0.490            0.490  0.000   0.000
sequence sequential               0.504            0.508  0.004   0.889
         distributional           0.526            0.568  0.043   8.085
         adversarial              0.465            0.419 -0.046  -9.851
         diffuse                  0.455            0.454 -0.001  -0.224
         relational               0.502            0.520  0.018   3.610
         Overall                  0.490            0.494  0.004   0.748
cwe      sequential               0.518            0.567  0.049   9.369
         distributional           0.483            0.524  0.041   8.538
         adversarial              0.336            0.296 -0.041 -12.113
         diffuse                  0.451            0.511  0.061  13.488
         relational               0.514            0.536  0.022   4.238
         Overall                  0.460            0.487  0.026   5.718
dwe      sequential               0.528            0.578  0.050   9.502
         distributional           0.542            0.574  0.033   6.061
         adversarial              0.327            0.288 -0.039 -11.950
         diffuse                  0.429            0.468  0.039   9.122
         relational               0.476            0.486  0.010   2.099
         Overall                  0.460            0.479  0.019   4.039
dcwe     sequential               0.520            0.569  0.049   9.427
         distributional           0.488            0.527  0.040   8.117
         adversarial              0.323            0.278 -0.045 -14.059
         diffuse                  0.451            0.510  0.059  13.001
         relational               0.512            0.533  0.021   4.008
         Overall                  0.459            0.483  0.024   5.332
random   sequential               0.512            0.512  0.000   0.000
         distributional           0.496            0.496  0.000   0.000
         adversarial              0.508            0.508  0.000   0.000
         diffuse                  0.500            0.500  0.000   0.000
         relational               0.496            0.496  0.000   0.000
         Overall                  0.502            0.502  0.000   0.000